# Trade Analysis — SQL (DuckDB)

Version SQL de `Analysis/notebooks/Notebook_Trade_Analysis.ipynb` — purement SQL, sans pandas ni plots.

Avant de construire la vraie chaîne d'analyse (`df_trade` → `df_concentration` → ...), ce notebook compare deux façons d'écrire des cellules SQL contre `data/sql/trade_analysis_sql.duckdb` :

- **Méthode A** — l'API Python de `duckdb` directement (`con.sql(...)`)
- **Méthode B** — l'extension `%%sql` (jupysql), où la cellule entière est du SQL pur

Les deux se connectent en **lecture seule** pour cette démo : à ce stade on ne fait qu'explorer, aucune raison d'écrire dans la base que les pipelines ETL alimentent.

## Méthode A — API Python `duckdb` directe

On ouvre une connexion, puis `con.sql("...")` exécute la requête et `.df()` convertit le résultat en DataFrame pandas (utile si on veut enchaîner avec autre chose ensuite).

In [1]:
import duckdb

DB_PATH = "../../../data/sql/trade_analysis_sql.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

con.sql("SELECT * FROM gold.entities LIMIT 5")

┌────────────┬────────────┬──────────────┬──────────────┬──────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────┐
│ entity_key │ country_id │ country_code │ country_iso3 │ country_name │ wb_country_code │ wb_country_iso3 │ wb_country_name │ entity_type │
│   int64    │  varchar   │    int64     │   varchar    │   varchar    │      int32      │     varchar     │     varchar     │   varchar   │
├────────────┼────────────┼──────────────┼──────────────┼──────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────┤
│          1 │ NG         │          566 │ NGA          │ Nigeria      │             566 │ NGA             │ Nigeria         │ COUNTRY     │
│          2 │ NO         │          579 │ NOR          │ Norway       │             578 │ NOR             │ Norway          │ COUNTRY     │
│          3 │ TH         │          764 │ THA          │ Thailand     │             764 │ THA             │ Thailand        │ COUNTRY     │
│          4 

In [2]:
# .df() donne un vrai DataFrame pandas si besoin de le reprendre ailleurs
df_sample = con.sql("SELECT * FROM gold.trade_data LIMIT 5").df()
df_sample

,trade_key,typeCode,reporter_key,partner_key,product_key,flow_key,year,primaryValue
0,1,C,33,134,97,1,2018,29679.825
1,2,C,33,134,97,1,2020,894.878
2,3,C,33,150,97,1,2021,100596.644
3,4,C,33,7,97,1,2021,113520.015
4,5,C,33,168,97,2,2016,1864246.599


In [3]:
# DuckDB refuse une 2e connexion au meme fichier avec une config differente --
# on ferme celle-ci avant que jupysql ouvre la sienne (methode B).
con.close()

## Méthode B — extension `%%sql` (jupysql)

Se charge une fois par notebook, puis on se connecte à la même base via une URL SQLAlchemy (`duckdb:///chemin`) — c'est `duckdb-engine` qui fournit ce dialecte. Ensuite chaque cellule `%%sql` est **entièrement** du SQL, sans wrapper Python — plus proche de l'expérience notebook SQL Server.

- `--alias conn` : un nom qu'on choisit pour cette connexion côté jupysql (comme le `AS persist` de nos `ATTACH` dans les pipelines) -- **pas** le nom réel de la base, qui reste le chemin du fichier.
- `--connection_arguments '{"read_only": true}'` : c'est CETTE option qui force réellement le mode lecture seule (équivalent du `read_only=True` de la méthode A) -- vérifié : une tentative de `CREATE TABLE` échoue bien avec *"attached in read-only mode"*.

**Méthode retenue pour la suite du notebook : B (`%%sql`).**

In [4]:
%load_ext sql
%sql duckdb:///../../../data/sql/trade_analysis_sql.duckdb --alias conn --connection_arguments '{"read_only": true}'

Connecting to 'conn'

In [5]:
%%sql
SELECT * FROM gold.entities LIMIT 5;

Running query in 'conn'

entity_key,country_id,country_code,country_iso3,country_name,wb_country_code,wb_country_iso3,wb_country_name,entity_type
1,NG,566,NGA,Nigeria,566,NGA,Nigeria,COUNTRY
2,NO,579,NOR,Norway,578,NOR,Norway,COUNTRY
3,TH,764,THA,Thailand,764,THA,Thailand,COUNTRY
4,VE,862,VEN,Venezuela,862,VEN,Venezuela,COUNTRY
5,AF,4,AFG,Afghanistan,4,AFG,Afghanistan,COUNTRY


On peut aussi récupérer le résultat d'une cellule `%%sql` dans une variable Python (utile si une étape suivante doit le réutiliser) :

In [6]:
%%sql result <<
SELECT * FROM gold.trade_data LIMIT 5;

Running query in 'conn'

In [7]:
df_from_magic = result.DataFrame()
df_from_magic

,trade_key,typeCode,reporter_key,partner_key,product_key,flow_key,year,primaryValue
0,1,C,33,134,97,1,2018,29679.825
1,2,C,33,134,97,1,2020,894.878
2,3,C,33,150,97,1,2021,100596.644
3,4,C,33,7,97,1,2021,113520.015
4,5,C,33,168,97,2,2016,1864246.599


<div style="background-color: #1a1a2e; padding: 20px; border-radius: 10px; text-align:center">

<h1 style="color:#f39c12">International Trade Analysis (2015–2024)</h1>
<h3 style="color:#aaaaaa">A Multi-Dimensional Framework for Country Performance & Bilateral Risk</h3>
</div>

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# A. INTRODUCTION
</div>

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# B. Executive Summary
</div>

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# C. Data & Methodology
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 1. Data Scope
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 2. Methodology Overview
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 3. Design Choices
</div>

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# D. Technical Setup
</div>

La connexion `conn` ci-dessus est en **lecture seule** — adaptée à la démo de la méthode, insuffisante à partir d'ici : la section E construit et persiste des tables d'analyse (`analysis.*`). On ferme `conn` (DuckDB refuse une 2e connexion au même fichier avec une config différente, comme vu plus haut) puis on ouvre une nouvelle connexion en **lecture-écriture**, active pour le reste du notebook.

In [8]:
from sql.connection import ConnectionManager
ConnectionManager.close_all()

In [9]:
%sql duckdb:///../../../data/sql/trade_analysis_sql.duckdb --alias db

Connecting and switching to connection 'db'

Formatage d'affichage global (K/M/B/T pour les grands montants, 2 décimales sinon) — repris de `_format_value` du notebook pandas. Réglé une seule fois ici, s'applique automatiquement à toutes les cellules `%%sql` suivantes : `SqlMagic.autopandas` fait que chaque résultat devient un vrai DataFrame pandas, et `pd.options.display.float_format` est un réglage global pandas qui s'applique à tout DataFrame affiché ensuite -- pas besoin de le répéter cellule par cellule. Les colonnes entières (`year`, `rank`, ...) ne sont pas concernées, seules les colonnes flottantes le sont.

In [10]:
%config SqlMagic.autopandas = True

import pandas as pd

def _format_value(x):
    if abs(x) >= 1e12: return f"{x/1e12:.2f}T"
    if abs(x) >= 1e9: return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6: return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3: return f"{x/1e3:.1f}K"
    return f"{x:.2f}"

pd.options.display.float_format = _format_value

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# E. Analytical Dataset Preparation
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 1. Exploring Gold Datasets
</div>

Pas de "chargement" à proprement parler — les tables gold sont déjà persistées. Pour chaque table : un `SUMMARIZE` filtré (type, min/max, count, % de nuls -- on garde l'essentiel, pas les quartiles/moyenne peu pertinents sur du texte) suivi d'un échantillon de 3 lignes.

In [1]:
%%sql
SELECT column_name, column_type, min, max, count, null_percentage
FROM (SUMMARIZE gold.entities);

UsageError: Cell magic `%%sql` not found.


In [ ]:
%%sql
SELECT * FROM gold.entities USING SAMPLE 3;

Running query in 'db'

,entity_key,country_id,country_code,country_iso3,country_name,wb_country_code,wb_country_iso3,wb_country_name,entity_type
0,72,NR,520,NRU,Nauru,520,NRU,Nauru,COUNTRY
1,210,TT,780,TTO,Trinidad and Tobago,780,TTO,Trinidad and Tobago,COUNTRY
2,298,XJ,<NA>,,,999,LAC,Latin America & Caribbean (excluding high income),ENTITY


In [ ]:
%%sql
SELECT column_name, column_type, min, max, count, null_percentage
FROM (SUMMARIZE gold.products);

Running query in 'db'

,column_name,column_type,min,max,count,null_percentage
0,product_key,BIGINT,1,97,97,0.00
1,cmdCode,VARCHAR,001,TOTAL,97,0.00
2,product_description,VARCHAR,AGGREGATION ALL PRODUCTS,Zinc and articles thereof,97,0.00
3,sector,VARCHAR,AGGREGATION ALL SECTORS,Transport,97,0.00


In [ ]:
%%sql
SELECT * FROM gold.products USING SAMPLE 3;

Running query in 'db'

,product_key,cmdCode,product_description,sector
0,71,071,"Natural or cultured pearls, precious or semi-p...",Bijoux et métaux précieux
1,81,082,"Tools, implements, cutlery",Machines et équipements
2,83,084,"Nuclear reactors, boilers, machinery",Machines et équipements


In [ ]:
%%sql
SELECT column_name, column_type, min, max, count, null_percentage
FROM (SUMMARIZE gold.trade_flow);

Running query in 'db'

,column_name,column_type,min,max,count,null_percentage
0,flow_key,BIGINT,1,4,4,0.00
1,flow_code,VARCHAR,M,X,4,0.00
2,flow_description,VARCHAR,EXPORT,RE_IMPORT,4,0.00


In [ ]:
%%sql
SELECT * FROM gold.trade_flow USING SAMPLE 3;

Running query in 'db'

,flow_key,flow_code,flow_description
0,2,X,EXPORT
1,3,RM,RE_IMPORT
2,4,RX,RE_EXPORT


In [ ]:
%%sql
SELECT column_name, column_type, min, max, count, null_percentage
FROM (SUMMARIZE gold.trade_data);

Running query in 'db'

,column_name,column_type,min,max,count,null_percentage
0,trade_key,BIGINT,1,543998,543998,0.00
1,typeCode,VARCHAR,C,C,543998,0.00
2,reporter_key,BIGINT,1,245,543998,0.00
3,partner_key,BIGINT,1,246,543998,0.00
4,product_key,BIGINT,97,97,543998,0.00
5,flow_key,BIGINT,1,2,543998,0.00
6,year,INTEGER,2015,2025,543998,0.00
7,primaryValue,DOUBLE,0.001,3593601435602.0,543998,0.00


In [ ]:
%%sql
SELECT * FROM gold.trade_data USING SAMPLE 3;

Running query in 'db'

,trade_key,typeCode,reporter_key,partner_key,product_key,flow_key,year,primaryValue
0,3331,C,227,63,97,2,2017,24.8K
1,7435,C,60,152,97,1,2025,778.58
2,492415,C,65,45,97,1,2016,25.1M


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 2. Merging Dimensions
</div>

Jointure de `gold.trade_data` avec `gold.entities` (deux fois : reporter et partenaire), `gold.trade_flow` et `gold.products`, puis regroupement/somme — fidèle à la version pandas (le `cmdCode` reste dans le `GROUP BY` comme en pandas, même s'il vaut toujours `TOTAL` en pratique). Résultat persisté dans un nouveau schéma `analysis`.

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS analysis;

Running query in 'db'

,Success


In [ ]:
%%sql
CREATE OR REPLACE TABLE analysis.trade AS
SELECT
    t.typeCode,
    rep.country_name,
    rep.country_code,
    rep.country_iso3,
    par.country_name AS partner_name,
    par.country_code AS partner_code,
    par.country_iso3 AS partner_iso3,
    t.year,
    f.flow_code,
    f.flow_description,
    p.cmdCode,
    p.product_description,
    p.sector,
    SUM(t.primaryValue) AS value
FROM gold.trade_data t
JOIN gold.entities rep ON t.reporter_key = rep.entity_key
JOIN gold.entities par ON t.partner_key  = par.entity_key
JOIN gold.trade_flow f ON t.flow_key     = f.flow_key
JOIN gold.products p   ON t.product_key  = p.product_key
GROUP BY
    t.typeCode, rep.country_name, rep.country_code, rep.country_iso3,
    par.country_name, par.country_code, par.country_iso3,
    t.year, f.flow_code, f.flow_description,
    p.cmdCode, p.product_description, p.sector;

Running query in 'db'

,Success


In [ ]:
%%sql
SELECT * FROM analysis.trade USING SAMPLE 3;

Running query in 'db'

,typeCode,country_name,country_code,country_iso3,partner_name,partner_code,partner_iso3,year,flow_code,flow_description,cmdCode,product_description,sector,value
0,C,Jordan,400,JOR,Italy,380,ITA,2019,M,IMPORT,TOTAL,AGGREGATION ALL PRODUCTS,AGGREGATION ALL SECTORS,608.8M
1,C,Niger,562,NER,Canada,124,CAN,2016,M,IMPORT,TOTAL,AGGREGATION ALL PRODUCTS,AGGREGATION ALL SECTORS,2.6M
2,C,Myanmar,104,MMR,Cameroon,120,CMR,2016,X,EXPORT,TOTAL,AGGREGATION ALL PRODUCTS,AGGREGATION ALL SECTORS,1.5M


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 3. Data Quality Checks
</div>

In [ ]:
%%sql
SELECT
    COUNT(*)                                     AS n_rows,
    MIN(year)                                    AS year_min,
    MAX(year)                                    AS year_max,
    COUNT(DISTINCT country_name)                 AS n_countries,
    COUNT(*) FILTER (WHERE value IS NULL)        AS missing_values,
    COUNT(*) FILTER (WHERE value < 0)            AS negative_values
FROM analysis.trade;

Running query in 'db'

,n_rows,year_min,year_max,n_countries,missing_values,negative_values
0,543998,2015,2025,188,0,0


Coverage : % de combinaisons (pays, année, flux) sans donnée.

In [ ]:
%%sql
WITH df_check AS (
    SELECT country_name, year, flow_code, COUNT(value) AS vals
    FROM analysis.trade
    GROUP BY country_name, year, flow_code
)
SELECT
    COUNT(*) FILTER (WHERE vals = 0)                          AS missing_combinations,
    COUNT(*)                                                  AS total_combinations,
    ROUND(100.0 * COUNT(*) FILTER (WHERE vals = 0) / COUNT(*), 1) AS missing_pct
FROM df_check;

Running query in 'db'

,missing_combinations,total_combinations,missing_pct
0,0,3475,0.00


## Comparaison rapide (démo méthodes)

| | Méthode A (`duckdb` direct) | Méthode B (`%%sql`) |
|---|---|---|
| Apparence de la cellule | Python + chaîne SQL | SQL pur (comme SQL Server) |
| Lecture seule | `read_only=True` | `--connection_arguments '{"read_only": true}'` |
| Récupérer un DataFrame | `.df()` direct | `%%sql result <<` puis `result.DataFrame()` |
| Multi-instructions dans une cellule | Oui (plusieurs `con.sql(...)`) | Oui, séparées par `;` |
| Dépendance supplémentaire | Aucune (déjà utilisé partout dans le projet) | `jupysql` (ajouté à `requirements.txt`) |

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# F. Global Trade Analysis
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 1. Analytical Base Tables
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 1.1 Table Enrichment
</div>

Filtre `analysis.trade` sur les échanges mondiaux (`partner_name='World'`, `cmdCode='TOTAL'`, `flow_code IN ('M','X')`, `typeCode='C'`, `year<=2024`), pivote M/X en colonnes, calcule les totaux mondiaux par année (pour les parts), balance, coverage ratio, parts import/export, rang export (`ROW_NUMBER` -- équivalent SQL de `rank(method="first")` en pandas : un rang unique par ligne, départage par ordre d'apparition, pas de rang partagé comme `RANK()`), puis les croissances annuelles par pays via `LAG()`.

In [24]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_by_country_year AS
WITH base AS (
    SELECT country_code, country_name, country_iso3, flow_code, year, value
    FROM analysis.trade
    WHERE partner_name = 'World'
      AND cmdCode = 'TOTAL'
      AND flow_code IN ('M', 'X')
      AND typeCode = 'C'
      AND year <= 2024
),
world_totals AS (
    SELECT year,
           SUM(value) FILTER (WHERE flow_code = 'M') AS m_sum,
           SUM(value) FILTER (WHERE flow_code = 'X') AS x_sum
    FROM base
    GROUP BY year
),
pivoted AS (
    SELECT country_code, country_name, country_iso3, year,
           SUM(value) FILTER (WHERE flow_code = 'M') AS M,
           SUM(value) FILTER (WHERE flow_code = 'X') AS X
    FROM base
    GROUP BY country_code, country_name, country_iso3, year
),
enriched AS (
    SELECT
        p.country_code, p.country_name, p.country_iso3, p.year,
        COALESCE(p.M, 0) AS M,
        COALESCE(p.X, 0) AS X,
        w.m_sum AS M_Sum,
        w.x_sum AS X_Sum,
        COALESCE(p.M, 0) + COALESCE(p.X, 0) AS total_trade,
        COALESCE(p.X, 0) - COALESCE(p.M, 0) AS trade_balance,
        CASE WHEN COALESCE(p.M, 0) != 0 THEN COALESCE(p.X, 0) / p.M ELSE 100 END AS coverage_ratio,
        COALESCE(p.M, 0) / w.m_sum * 100 AS import_share,
        COALESCE(p.X, 0) / w.x_sum * 100 AS export_share,
        (COALESCE(p.X, 0) - COALESCE(p.M, 0)) / 1e9 AS trade_balance_B
    FROM pivoted p
    JOIN world_totals w ON p.year = w.year
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY year ORDER BY export_share DESC) AS rank,
    (M / NULLIF(LAG(M) OVER (PARTITION BY country_name ORDER BY year), 0) - 1) * 100 AS import_growth,
    (X / NULLIF(LAG(X) OVER (PARTITION BY country_name ORDER BY year), 0) - 1) * 100 AS export_growth,
    (trade_balance_B / NULLIF(LAG(trade_balance_B) OVER (PARTITION BY country_name ORDER BY year), 0) - 1) * 100 AS balance_growth,
    (export_share / NULLIF(LAG(export_share) OVER (PARTITION BY country_name ORDER BY year), 0) - 1) * 100 AS export_share_growth,
    (import_share / NULLIF(LAG(import_share) OVER (PARTITION BY country_name ORDER BY year), 0) - 1) * 100 AS import_share_growth
FROM enriched
ORDER BY country_name, year;

Running query in 'db'

,Success


In [25]:
%%sql
SELECT * FROM analysis.trade_by_country_year USING SAMPLE 3;

Running query in 'db'

,country_code,country_name,country_iso3,year,M,X,M_Sum,X_Sum,total_trade,trade_balance,coverage_ratio,import_share,export_share,trade_balance_B,rank,import_growth,export_growth,balance_growth,export_share_growth,import_share_growth
0,300,Greece,GRC,2018,65.14B,39.49B,19.04T,18.67T,104.63B,-25.65B,0.61,0.34,0.21,-25.65,57,17.80,22.81,10.82,11.77,7.01
1,508,Mozambique,MOZ,2024,9.21B,8.21B,22.08T,21.47T,17.42B,-1.00B,0.89,0.04,0.04,-1.00,89,-8.71,-0.78,-44.86,2.45,-5.82
2,752,Sweden,SWE,2020,149.44B,154.94B,17.08T,16.84T,304.37B,5.50B,1.04,0.87,0.92,5.50,32,-5.99,-3.49,249.59,3.90,1.74


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 1.2 Dynamic Features
</div>

Par pays : totaux, croissance moyenne, CAGR (première/dernière valeur ordonnées par année via `FIRST(x ORDER BY year ASC/DESC)`, `NULL` si valeur de départ ou d'arrivée <= 0), volatilité (écart-type des croissances), fréquence d'années à croissance négative (dénominateur = **toutes** les années du pays, pas seulement celles avec une croissance calculable -- comme en pandas où `(NaN < 0)` vaut `False`, pas `NaN`, et compte donc dans la moyenne). Puis normalisation z-score et score de résilience (`z_cagr - z_volatility - z_negfreq`).

In [26]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_dynamic AS
WITH bounds AS (
    SELECT
        country_name,
        FIRST(X ORDER BY year ASC)  AS x_start,
        FIRST(X ORDER BY year DESC) AS x_end,
        FIRST(M ORDER BY year ASC)  AS m_start,
        FIRST(M ORDER BY year DESC) AS m_end,
        COUNT(*) AS n_years
    FROM analysis.trade_by_country_year
    GROUP BY country_name
),
aggregates AS (
    SELECT
        country_name,
        SUM(X) AS total_export,
        SUM(M) AS total_import,
        AVG(export_growth) AS avg_growth_export,
        AVG(import_growth) AS avg_growth_import,
        STDDEV(export_growth) AS volatility_export,
        STDDEV(import_growth) AS volatility_import,
        COUNT(export_growth) AS years_export,
        COUNT(import_growth) AS years_import,
        COUNT(*) FILTER (WHERE export_growth < 0)::DOUBLE / COUNT(*) AS negfreq_export,
        COUNT(*) FILTER (WHERE import_growth < 0)::DOUBLE / COUNT(*) AS negfreq_import
    FROM analysis.trade_by_country_year
    GROUP BY country_name
),
combined AS (
    SELECT
        a.country_name,
        a.total_export, a.total_import,
        a.avg_growth_export, a.avg_growth_import,
        CASE WHEN b.n_years >= 2 AND b.x_start > 0 AND b.x_end > 0
             THEN POWER(b.x_end / b.x_start, 1.0 / (b.n_years - 1)) - 1
             ELSE NULL END AS cagr_export,
        CASE WHEN b.n_years >= 2 AND b.m_start > 0 AND b.m_end > 0
             THEN POWER(b.m_end / b.m_start, 1.0 / (b.n_years - 1)) - 1
             ELSE NULL END AS cagr_import,
        a.volatility_export, a.volatility_import,
        a.years_export, a.years_import,
        a.negfreq_export, a.negfreq_import
    FROM aggregates a
    JOIN bounds b ON a.country_name = b.country_name
),
zscored AS (
    SELECT
        *,
        (cagr_export - AVG(cagr_export) OVER ()) / STDDEV(cagr_export) OVER () AS z_cagr_export,
        (cagr_import - AVG(cagr_import) OVER ()) / STDDEV(cagr_import) OVER () AS z_cagr_import,
        (volatility_export - AVG(volatility_export) OVER ()) / STDDEV(volatility_export) OVER () AS z_volatility_export,
        (volatility_import - AVG(volatility_import) OVER ()) / STDDEV(volatility_import) OVER () AS z_volatility_import,
        (negfreq_export - AVG(negfreq_export) OVER ()) / STDDEV(negfreq_export) OVER () AS z_negfreq_export,
        (negfreq_import - AVG(negfreq_import) OVER ()) / STDDEV(negfreq_import) OVER () AS z_negfreq_import
    FROM combined
)
SELECT
    *,
    z_cagr_export - z_volatility_export - z_negfreq_export AS resilience_export,
    z_cagr_import - z_volatility_import - z_negfreq_import AS resilience_import
FROM zscored
ORDER BY resilience_export DESC;

Running query in 'db'

,Success


In [27]:
%%sql
SELECT * FROM analysis.trade_dynamic USING SAMPLE 3;

Running query in 'db'

,country_name,total_export,total_import,avg_growth_export,avg_growth_import,cagr_export,cagr_import,volatility_export,volatility_import,years_export,...,negfreq_export,negfreq_import,z_cagr_export,z_cagr_import,z_volatility_export,z_volatility_import,z_negfreq_export,z_negfreq_import,resilience_export,resilience_import
0,Nicaragua,58.31B,85.38B,5.81,9.03,0.05,0.08,9.62,18.94,9,...,0.20,0.40,0.02,0.40,-0.31,0.10,-0.95,0.46,1.27,-0.16
1,Rep. of Moldova,29.47B,64.00B,8.01,10.44,0.07,0.10,17.41,14.90,9,...,0.30,0.20,0.18,0.68,-0.20,-0.32,-0.24,-1.05,0.61,2.04
2,Belarus,215.19B,244.60B,8.47,6.79,0.07,0.06,20.06,17.80,6,...,0.43,0.29,0.20,0.07,-0.16,-0.01,0.67,-0.40,-0.32,0.49


**Écart volontaire avec pandas** : quand un pays part d'une valeur nulle une année puis reprend un échange l'année suivante (ex. Cayman Isds : `X=0` en 2015, `X=22M` en 2020, années 2016-2019 absentes), `pct_change()` en pandas donne `inf` (division par zéro), qui contamine ensuite la moyenne/écart-type/z-score de ce pays en `NaN`. Ici, `NULLIF` transforme ce cas en `NULL`, proprement exclu des agrégats -- le pays garde une volatilité et une résilience calculables au lieu de disparaître à cause d'un seul point de départ à zéro. Conséquence : `z_volatility_export`/`resilience_export` diffèrent très légèrement (~0.02 max) de pandas pour **tous** les pays, la moyenne/écart-type global se recalculant sur un pays de plus (Cayman Isds) que dans la version pandas -- vérifié, tout le reste (totaux, CAGR, volatilité brute, z-scores restants) est identique.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 2. Global Trade System
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 2.2 Global Trade Evolution
</div>

Analyse le commerce mondial comme un système (tous pays confondus), pas pays par pays. Remplace le graphique pandas par deux tables : l'évolution année par année (10 lignes) et un résumé CAGR 2015→2024 par métrique (3 lignes) -- la première montre la trajectoire, la seconde la synthétise, ensemble elles portent la même information que le graphique.

In [28]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_evolution AS
SELECT
    year,
    SUM(total_trade) AS total_trade,
    SUM(X) AS total_export,
    SUM(M) AS total_import,
    (SUM(total_trade) / NULLIF(LAG(SUM(total_trade)) OVER (ORDER BY year), 0) - 1) * 100 AS yoy_total_trade_pct,
    (SUM(X) / NULLIF(LAG(SUM(X)) OVER (ORDER BY year), 0) - 1) * 100 AS yoy_export_pct,
    (SUM(M) / NULLIF(LAG(SUM(M)) OVER (ORDER BY year), 0) - 1) * 100 AS yoy_import_pct
FROM analysis.trade_by_country_year
GROUP BY year
ORDER BY year;

Running query in 'db'

,Success


In [29]:
%%sql
SELECT * FROM analysis.trade_evolution;

Running query in 'db'

,year,total_trade,total_export,total_import,yoy_total_trade_pct,yoy_export_pct,yoy_import_pct
0,2015,31.90T,15.87T,16.03T,NaN,NaN,NaN
1,2016,31.08T,15.45T,15.63T,-2.57,-2.66,-2.48
2,2017,34.29T,16.99T,17.30T,10.32,10.00,10.64
3,2018,37.71T,18.67T,19.04T,9.98,9.88,10.08
4,2019,36.61T,18.12T,18.49T,-2.91,-2.92,-2.90
5,2020,33.92T,16.84T,17.08T,-7.35,-7.11,-7.59
6,2021,42.78T,21.21T,21.57T,26.13,25.97,26.28
7,2022,47.32T,23.11T,24.21T,10.61,8.97,12.23
8,2023,44.94T,22.16T,22.78T,-5.02,-4.09,-5.91
9,2024,43.55T,21.47T,22.08T,-3.11,-3.15,-3.07


In [30]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_evolution_summary AS
WITH unpivoted AS (
    UNPIVOT (SELECT year, total_trade, total_export, total_import FROM analysis.trade_evolution)
    ON total_trade, total_export, total_import
    INTO NAME metric VALUE value
)
SELECT
    metric,
    FIRST(value ORDER BY year ASC)  AS value_2015,
    FIRST(value ORDER BY year DESC) AS value_2024,
    (POWER(FIRST(value ORDER BY year DESC) / FIRST(value ORDER BY year ASC), 1.0 / 9) - 1) * 100 AS cagr_pct
FROM unpivoted
GROUP BY metric;

Running query in 'db'

,Success


In [31]:
%%sql
SELECT * FROM analysis.trade_evolution_summary;

Running query in 'db'

,metric,value_2015,value_2024,cagr_pct
0,total_trade,31.90T,43.55T,3.52
1,total_import,16.03T,22.08T,3.62
2,total_export,15.87T,21.47T,3.41


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 2.3 Global Trade Distribution
</div>

Top 15 pays par export et par import, année 2024 (comme en pandas). Part = valeur du pays / total mondial de cette année pour cette métrique (pas le total des 15 -- les parts ne totalisent donc pas 100%). Part cumulée = somme courante ordonnée par rang.

In [32]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_top_countries AS
WITH unpivoted AS (
    UNPIVOT (SELECT country_name, X, M FROM analysis.trade_by_country_year WHERE year = 2024)
    ON X, M
    INTO NAME metric VALUE value
),
totals AS (
    SELECT metric, SUM(value) AS world_total
    FROM unpivoted
    GROUP BY metric
),
ranked AS (
    SELECT
        u.metric, u.country_name, u.value,
        ROW_NUMBER() OVER (PARTITION BY u.metric ORDER BY u.value DESC) AS rank,
        u.value / t.world_total * 100 AS share_pct
    FROM unpivoted u
    JOIN totals t ON u.metric = t.metric
)
SELECT
    metric, rank, country_name, value, share_pct,
    SUM(share_pct) OVER (PARTITION BY metric ORDER BY rank) AS cumulative_share_pct
FROM ranked
WHERE rank <= 15
ORDER BY metric, rank;

Running query in 'db'

,Success


In [33]:
%%sql
SELECT * FROM analysis.trade_top_countries WHERE metric = 'X' ORDER BY rank;

Running query in 'db'

,metric,rank,country_name,value,share_pct,cumulative_share_pct
0,X,1,China,3.58T,16.66,16.66
1,X,2,USA,2.06T,9.61,26.28
2,X,3,Germany,1.63T,7.60,33.87
3,X,4,Japan,707.39B,3.30,37.17
4,X,5,Netherlands,699.26B,3.26,40.43
5,X,6,Rep. of Korea,683.13B,3.18,43.61
6,X,7,Italy,652.41B,3.04,46.65
7,X,8,France,645.96B,3.01,49.66
8,X,9,"China, Hong Kong SAR",639.41B,2.98,52.64
9,X,10,Mexico,619.66B,2.89,55.52


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 2.4 Global Trade Balance Structure
</div>

Top 10 pays en surplus et top 10 en déficit commercial, année 2024 (comme en pandas).

In [34]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_balance_top AS
WITH year_data AS (
    SELECT country_name, trade_balance
    FROM analysis.trade_by_country_year
    WHERE year = 2024
),
surplus AS (
    SELECT 'surplus' AS category, country_name, trade_balance,
           ROW_NUMBER() OVER (ORDER BY trade_balance DESC) AS rank
    FROM year_data
    QUALIFY rank <= 10
),
deficit AS (
    SELECT 'deficit' AS category, country_name, trade_balance,
           ROW_NUMBER() OVER (ORDER BY trade_balance ASC) AS rank
    FROM year_data
    QUALIFY rank <= 10
)
SELECT * FROM surplus
UNION ALL
SELECT * FROM deficit
ORDER BY category, rank;

Running query in 'db'

,Success


In [35]:
%%sql
SELECT * FROM analysis.trade_balance_top ORDER BY category, rank;

Running query in 'db'

,category,country_name,trade_balance,rank
0,deficit,USA,-1.29T,1
1,deficit,United Kingdom,-293.18B,2
2,deficit,India,-263.31B,3
3,deficit,France,-116.09B,4
4,deficit,Turkiye,-82.23B,5
5,deficit,Philippines,-61.90B,6
6,deficit,"China, Hong Kong SAR",-58.49B,7
7,deficit,Egypt,-49.85B,8
8,deficit,Spain,-47.60B,9
9,deficit,Romania,-36.15B,10


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 2.5 Global Trade Concentration
</div>

#### 2.5.1 Top3 / Top5 / Top10 Concentration

In [36]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_concentration AS
SELECT
    year,
    SUM(export_share) FILTER (WHERE rank <= 3)  AS top3_share,
    SUM(export_share) FILTER (WHERE rank <= 5)  AS top5_share,
    SUM(export_share) FILTER (WHERE rank <= 10) AS top10_share
FROM analysis.trade_by_country_year
GROUP BY year
ORDER BY year;

Running query in 'db'

,Success


#### 2.5.2 HHI

In [37]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_hhi AS
SELECT
    year,
    SUM(export_share * export_share) AS hhi_global
FROM analysis.trade_by_country_year
GROUP BY year
ORDER BY year;

Running query in 'db'

,Success


#### 2.5.3 Global Trade Hierarchy Stability (Rank Correlation)

Corrélation de Spearman entre le rang export d'une année et celui de l'année précédente, par pays. Comme la corrélation de Spearman est la corrélation de Pearson calculée sur les rangs (pas les valeurs brutes) -- et qu'on a déjà `rank` -- `CORR(rank_t, rank_t-1)` natif DuckDB donne directement le résultat, sans fonction spécifique.

In [38]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_hierarchy_stability AS
SELECT
    t.year,
    CORR(t.rank, t1.rank) AS spearman_correlation
FROM analysis.trade_by_country_year t
JOIN analysis.trade_by_country_year t1
    ON t.country_name = t1.country_name AND t1.year = t.year - 1
GROUP BY t.year
ORDER BY t.year;

Running query in 'db'

,Success


#### 2.5.4 Concentration Synthesis

In [39]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_concentration_summary AS
SELECT
    c.year, c.top3_share, c.top5_share, c.top10_share,
    h.hhi_global,
    s.spearman_correlation
FROM analysis.trade_concentration c
LEFT JOIN analysis.trade_hhi h ON c.year = h.year
LEFT JOIN analysis.trade_hierarchy_stability s ON c.year = s.year
ORDER BY c.year;

Running query in 'db'

,Success


In [40]:
%%sql
SELECT * FROM analysis.trade_concentration_summary;

Running query in 'db'

,year,top3_share,top5_share,top10_share,hhi_global,spearman_correlation
0,2015,32.16,39.42,54.50,494.06,NaN
1,2016,31.63,39.15,54.21,479.44,1.00
2,2017,30.84,38.32,53.33,464.15,1.00
3,2018,30.61,37.80,52.62,459.07,1.00
4,2019,31.10,38.18,52.75,471.19,1.00
5,2020,32.11,39.19,53.86,505.20,1.00
6,2021,31.62,38.47,52.58,498.62,1.00
7,2022,31.81,38.37,52.15,499.69,1.00
8,2023,32.14,38.62,52.79,502.21,1.00
9,2024,33.87,40.43,55.52,554.29,1.00


**Écart volontaire avec pandas** : `spearman_correlation` diffère très légèrement (max ~0.22%) car `scipy.stats.spearmanr` re-classe ses arguments en interne sur le sous-ensemble de pays **communs** aux deux années comparées, alors que `CORR(rank_t, rank_t1)` utilise ici les rangs **originaux** (calculés sur l'ensemble des pays de chaque année, qui varie -- 175 pays en 2015, 136 en 2024). Écart jugé négligeable pour l'interprétation (stabilité de la hiérarchie commerciale) -- tout le reste (`top3/5/10_share`, `hhi_global`) est identique à pandas au bruit flottant près.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 2.6 Global Trade Volatility
</div>

#### 2.6.1 Volatility

Pas de nouveau calcul -- `volatility_export`/`volatility_import` sont déjà dans `analysis.trade_dynamic` (F.1.2). Seule la carte du monde et la matrice de dispersion manquent (rendu visuel pur).

#### 2.6.2 Countries risk (volatility vs growth)

Filtre `years_export >= 5` + valeurs non nulles, puis classification en 4 quadrants selon les médianes de croissance moyenne et de volatilité.

In [41]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_risk_quadrant AS
WITH filtered AS (
    SELECT country_name, avg_growth_export, volatility_export, total_export
    FROM analysis.trade_dynamic
    WHERE years_export >= 5
      AND avg_growth_export IS NOT NULL
      AND volatility_export IS NOT NULL
      AND total_export IS NOT NULL
),
medians AS (
    SELECT
        MEDIAN(avg_growth_export) AS median_growth,
        MEDIAN(volatility_export) AS median_vol
    FROM filtered
)
SELECT
    f.country_name, f.avg_growth_export, f.volatility_export, f.total_export,
    CASE
        WHEN f.avg_growth_export >= m.median_growth AND f.volatility_export <= m.median_vol THEN 'Stable Stars'
        WHEN f.avg_growth_export >= m.median_growth AND f.volatility_export >  m.median_vol THEN 'Boom-Bust'
        WHEN f.avg_growth_export <  m.median_growth AND f.volatility_export <= m.median_vol THEN 'Mature'
        ELSE 'Vulnerable'
    END AS quadrant
FROM filtered f
CROSS JOIN medians m
ORDER BY quadrant, country_name;

Running query in 'db'

,Success


In [42]:
%%sql
SELECT quadrant, COUNT(*) AS n_countries FROM analysis.trade_risk_quadrant GROUP BY quadrant ORDER BY quadrant;

Running query in 'db'

,quadrant,n_countries
0,Boom-Bust,57
1,Mature,57
2,Stable Stars,27
3,Vulnerable,26


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 3. Country Trade Structure
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.1 Country Data
</div>

Le focus pays se paramètre avec une variable native DuckDB (`SET VARIABLE`), lue ensuite par `getvariable('country')` dans toutes les requêtes de cette section -- un seul endroit à modifier pour changer de pays. Table équivalente à `df_country` (pandas) : `analysis.trade_by_country_year` filtrée sur ce pays, déjà construite en F.1.1.

In [43]:
%%sql
-- Pays de démo -- changer ici pour reparamétrer toute la section 3
SET VARIABLE country = 'USA';

Running query in 'db'

,Success


In [44]:
%%sql
SELECT *
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
ORDER BY year;

Running query in 'db'

,country_code,country_name,country_iso3,year,M,X,M_Sum,X_Sum,total_trade,trade_balance,coverage_ratio,import_share,export_share,trade_balance_B,rank,import_growth,export_growth,balance_growth,export_share_growth,import_share_growth
0,842,USA,USA,2015,2.31T,1.50T,16.03T,15.87T,3.82T,-811.58B,0.65,14.43,9.46,-811.58,2,NaN,NaN,NaN,NaN,NaN
1,842,USA,USA,2016,2.25T,1.45T,15.63T,15.45T,3.70T,-796.26B,0.65,14.37,9.39,-796.26,2,-2.86,-3.39,-1.89,-0.75,-0.39
2,842,USA,USA,2017,2.41T,1.55T,17.30T,16.99T,3.95T,-859.47B,0.64,13.91,9.10,-859.47,2,7.04,6.54,7.94,-3.15,-3.26
3,842,USA,USA,2018,2.61T,1.67T,19.04T,18.67T,4.28T,-946.13B,0.64,13.72,8.92,-946.13,2,8.57,7.73,10.08,-1.96,-1.37
4,842,USA,USA,2019,2.57T,1.64T,18.49T,18.12T,4.21T,-923.22B,0.64,13.89,9.07,-923.22,2,-1.68,-1.26,-2.42,1.71,1.25
5,842,USA,USA,2020,2.41T,1.43T,17.08T,16.84T,3.84T,-975.13B,0.59,14.08,8.50,-975.13,2,-6.31,-13.02,5.62,-6.36,1.39
6,842,USA,USA,2021,2.93T,1.75T,21.57T,21.21T,4.69T,-1.18T,0.60,13.59,8.27,-1.2K,2,21.93,22.58,20.99,-2.69,-3.44
7,842,USA,USA,2022,3.37T,2.06T,24.21T,23.11T,5.43T,-1.31T,0.61,13.93,8.92,-1.3K,2,15.00,17.62,11.10,7.94,2.47
8,842,USA,USA,2023,3.17T,2.02T,22.78T,22.16T,5.19T,-1.15T,0.64,13.91,9.11,-1.1K,2,-6.06,-2.11,-12.27,2.06,-0.16
9,842,USA,USA,2024,3.36T,2.06T,22.08T,21.47T,5.42T,-1.29T,0.61,15.20,9.61,-1.3K,2,5.94,2.24,12.44,5.57,9.30


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.2 Export / Import Trends
</div>

In [45]:
%%sql
SELECT
    year, X, M, trade_balance,
    CASE WHEN trade_balance >= 0 THEN 'Surplus' ELSE 'Deficit' END AS zone
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
ORDER BY year;

Running query in 'db'

,year,X,M,trade_balance,zone
0,2015,1.50T,2.31T,-811.58B,Deficit
1,2016,1.45T,2.25T,-796.26B,Deficit
2,2017,1.55T,2.41T,-859.47B,Deficit
3,2018,1.67T,2.61T,-946.13B,Deficit
4,2019,1.64T,2.57T,-923.22B,Deficit
5,2020,1.43T,2.41T,-975.13B,Deficit
6,2021,1.75T,2.93T,-1.18T,Deficit
7,2022,2.06T,3.37T,-1.31T,Deficit
8,2023,2.02T,3.17T,-1.15T,Deficit
9,2024,2.06T,3.36T,-1.29T,Deficit


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.3 Export Share / Import Share Trends
</div>

In [46]:
%%sql
SELECT year, import_share, export_share
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
ORDER BY year;

Running query in 'db'

,year,import_share,export_share
0,2015,14.43,9.46
1,2016,14.37,9.39
2,2017,13.91,9.10
3,2018,13.72,8.92
4,2019,13.89,9.07
5,2020,14.08,8.50
6,2021,13.59,8.27
7,2022,13.93,8.92
8,2023,13.91,9.11
9,2024,15.20,9.61


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.4 Trade Balance
</div>

In [47]:
%%sql
SELECT year, trade_balance, coverage_ratio
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
ORDER BY year;

Running query in 'db'

,year,trade_balance,coverage_ratio
0,2015,-811.58B,0.65
1,2016,-796.26B,0.65
2,2017,-859.47B,0.64
3,2018,-946.13B,0.64
4,2019,-923.22B,0.64
5,2020,-975.13B,0.59
6,2021,-1.18T,0.60
7,2022,-1.31T,0.61
8,2023,-1.15T,0.64
9,2024,-1.29T,0.61


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.5 Trade Growth
</div>

In [48]:
%%sql
SELECT year, export_growth, import_growth
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
  AND export_growth IS NOT NULL
ORDER BY year;

Running query in 'db'

,year,export_growth,import_growth
0,2016,-3.39,-2.86
1,2017,6.54,7.04
2,2018,7.73,8.57
3,2019,-1.26,-1.68
4,2020,-13.02,-6.31
5,2021,22.58,21.93
6,2022,17.62,15.00
7,2023,-2.11,-6.06
8,2024,2.24,5.94


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.6 Global Trade View
</div>

Pas de nouvelle donnée ici -- en pandas c'est un dashboard 4 panneaux combinant visuellement 3.2 (import/export), 3.3 (parts), 3.5 (croissance) et 3.4 (balance). Les quatre tables ci-dessus couvrent déjà l'intégralité de cette information.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 3.7 Structural Indicators
</div>

#### 3.7.1 Create Structural Table

Base = `analysis.trade_by_country_year` (médiane par pays de `export_share`/`import_share`/`coverage_ratio` + leurs z-scores, calculés sur l'ensemble des pays), jointe à `analysis.trade_dynamic` (`cagr_export`, `volatility_export`, `negfreq_export` et leurs z-scores, déjà calculés en F.1.2) -- pays sans CAGR/volatilité calculable exclus (`WHERE ... IS NOT NULL`, équivalent du `dropna` pandas après le merge). Contrairement à pandas (deux passes : classification puis score), tout est fait ici en une seule requête -- le score composite et les 5 profils ne dépendent que de colonnes déjà calculées dans les CTE précédentes, pas besoin d'étapes séparées.

In [49]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_structural AS
WITH aggregated AS (
    SELECT
        country_name,
        MEDIAN(export_share)   AS export_share_structural,
        MEDIAN(import_share)   AS import_share_structural,
        MEDIAN(coverage_ratio) AS coverage_ratio_structural
    FROM analysis.trade_by_country_year
    GROUP BY country_name
),
zscored AS (
    SELECT
        *,
        (export_share_structural - AVG(export_share_structural) OVER ()) / STDDEV(export_share_structural) OVER ()     AS z_export_share,
        (import_share_structural - AVG(import_share_structural) OVER ()) / STDDEV(import_share_structural) OVER ()     AS z_import_share,
        (coverage_ratio_structural - AVG(coverage_ratio_structural) OVER ()) / STDDEV(coverage_ratio_structural) OVER () AS z_coverage_ratio
    FROM aggregated
),
joined AS (
    SELECT
        z.*,
        d.cagr_export, d.cagr_import, d.volatility_export, d.volatility_import,
        d.negfreq_export, d.negfreq_import,
        d.z_cagr_export, d.z_volatility_export, d.z_negfreq_export
    FROM zscored z
    JOIN analysis.trade_dynamic d ON z.country_name = d.country_name
    WHERE d.cagr_export IS NOT NULL
      AND d.volatility_export IS NOT NULL
      AND d.negfreq_export IS NOT NULL
),
scored AS (
    SELECT
        *,
        z_export_share * 0.60 + z_coverage_ratio * 0.10 + z_cagr_export * 0.15
            - z_volatility_export * 0.10 - z_negfreq_export * 0.05 AS structural_score
    FROM joined
),
classified AS (
    SELECT
        *,
        CASE
            WHEN export_share_structural >= 2   THEN 'Leader'
            WHEN export_share_structural >= 0.5 THEN 'Major'
            WHEN export_share_structural >= 0.1 THEN 'Mid'
            ELSE 'Small'
        END AS size_profile,
        CASE
            WHEN coverage_ratio_structural >= 1.15 THEN 'Strong Surplus'
            WHEN coverage_ratio_structural >= 1    THEN 'Slight Surplus'
            WHEN coverage_ratio_structural >= 0.85 THEN 'Slight Deficit'
            ELSE 'Structural Deficit'
        END AS balance_profile,
        CASE
            WHEN z_cagr_export >= 1  THEN 'Strong Grower'
            WHEN z_cagr_export >= 0  THEN 'Moderate Grower'
            WHEN z_cagr_export >= -1 THEN 'Weak Grower'
            ELSE 'Declining'
        END AS growth_profile,
        CASE
            WHEN z_volatility_export <= -0.5 THEN 'Stable'
            WHEN z_volatility_export <= 0.5  THEN 'Moderate'
            WHEN z_volatility_export <= 1.5  THEN 'Volatile'
            ELSE 'Very Volatile'
        END AS stability_profile,
        CASE
            WHEN negfreq_export <= 0.20 THEN 'Very Consistent'
            WHEN negfreq_export <= 0.35 THEN 'Consistent'
            WHEN negfreq_export <= 0.55 THEN 'Inconsistent'
            ELSE 'Very Inconsistent'
        END AS consistency_profile,
        QUANTILE_CONT(structural_score, 0.85) OVER () AS q85,
        QUANTILE_CONT(structural_score, 0.50) OVER () AS q50,
        QUANTILE_CONT(structural_score, 0.15) OVER () AS q15
    FROM scored
)
SELECT
    country_name,
    export_share_structural, import_share_structural, coverage_ratio_structural,
    z_export_share, z_import_share, z_coverage_ratio,
    cagr_export, cagr_import, volatility_export, volatility_import,
    negfreq_export, negfreq_import, z_cagr_export, z_volatility_export, z_negfreq_export,
    size_profile, balance_profile, growth_profile, stability_profile, consistency_profile,
    structural_score,
    CASE
        WHEN structural_score >= q85 THEN 'Strong'
        WHEN structural_score >= q50 THEN 'Good'
        WHEN structural_score >= q15 THEN 'Moderate'
        ELSE 'Weak'
    END AS structural_profile
FROM classified
ORDER BY structural_score DESC;

Running query in 'db'

,Success


In [50]:
%%sql
SELECT * FROM analysis.trade_structural USING SAMPLE 3;

Running query in 'db'

,country_name,export_share_structural,import_share_structural,coverage_ratio_structural,z_export_share,z_import_share,z_coverage_ratio,cagr_export,cagr_import,volatility_export,...,z_cagr_export,z_volatility_export,z_negfreq_export,size_profile,balance_profile,growth_profile,stability_profile,consistency_profile,structural_score,structural_profile
0,Spain,1.85,2.00,0.90,0.84,0.95,-0.05,0.04,0.04,10.14,...,-0.13,-0.30,0.47,Major,Slight Deficit,Weak Grower,Moderate,Inconsistent,0.49,Strong
1,Indonesia,0.98,0.92,1.10,0.28,0.25,-0.02,0.07,0.06,17.46,...,0.15,-0.20,0.47,Major,Slight Surplus,Moderate Grower,Moderate,Inconsistent,0.19,Good
2,Greece,0.21,0.34,0.60,-0.22,-0.13,-0.09,0.07,0.07,15.55,...,0.24,-0.22,1.18,Mid,Structural Deficit,Moderate Grower,Moderate,Inconsistent,-0.14,Moderate


#### 3.7.2 Scoring

`structural_score` et `structural_profile` sont déjà dans la table (calculés dans la même requête que 3.7.1). Vérification : top 3 pays par score, et les seuils de quantile utilisés pour la classification.

In [51]:
%%sql
SELECT country_name, structural_score, structural_profile
FROM analysis.trade_structural
ORDER BY structural_score DESC
LIMIT 3;

Running query in 'db'

,country_name,structural_score,structural_profile
0,China,5.60,Strong
1,USA,3.28,Strong
2,Germany,2.97,Strong


In [52]:
%%sql
SELECT
    QUANTILE_CONT(structural_score, 0.85) AS q85,
    QUANTILE_CONT(structural_score, 0.50) AS q50,
    QUANTILE_CONT(structural_score, 0.15) AS q15
FROM analysis.trade_structural;

Running query in 'db'

,q85,q50,q15
0,0.29,-0.14,-0.33


#### 3.7.3 Summary

Distribution des 6 profils (comptage par catégorie, une table au lieu d'une boucle de `value_counts()`) + statistiques du score composite.

In [53]:
%%sql
WITH unpivoted AS (
    UNPIVOT (
        SELECT size_profile, balance_profile, growth_profile, stability_profile, consistency_profile, structural_profile
        FROM analysis.trade_structural
    )
    ON size_profile, balance_profile, growth_profile, stability_profile, consistency_profile, structural_profile
    INTO NAME dimension VALUE profile
)
SELECT dimension, profile, COUNT(*) AS n_countries
FROM unpivoted
GROUP BY dimension, profile
ORDER BY dimension, n_countries DESC;

Running query in 'db'

,dimension,profile,n_countries
0,balance_profile,Structural Deficit,108
1,balance_profile,Strong Surplus,28
2,balance_profile,Slight Deficit,24
3,balance_profile,Slight Surplus,23
4,consistency_profile,Inconsistent,76
5,consistency_profile,Consistent,62
6,consistency_profile,Very Consistent,37
7,consistency_profile,Very Inconsistent,8
8,growth_profile,Weak Grower,89
9,growth_profile,Moderate Grower,67


In [54]:
%%sql
SELECT
    MIN(structural_score)    AS min_score,
    MAX(structural_score)    AS max_score,
    AVG(structural_score)    AS avg_score,
    STDDEV(structural_score) AS stddev_score,
    MEDIAN(structural_score) AS median_score
FROM analysis.trade_structural;

Running query in 'db'

,min_score,max_score,avg_score,stddev_score,median_score
0,-1.38,5.60,-0.01,0.64,-0.14


#### 3.7.4 Radar Chart

Remplacé par une table : chaque profil converti en niveau numérique (1.0 = meilleur, 0.0 = pire, comme l'échelle du radar pandas), pour quelques pays comparés à la moyenne mondiale sur les 5 dimensions -- même information que le radar, sans le graphique.

In [55]:
%%sql
WITH levels AS (
    SELECT
        country_name,
        CASE size_profile WHEN 'Leader' THEN 1.0 WHEN 'Major' THEN 0.67 WHEN 'Mid' THEN 0.33 ELSE 0.0 END AS size_level,
        CASE balance_profile WHEN 'Strong Surplus' THEN 1.0 WHEN 'Slight Surplus' THEN 0.67 WHEN 'Slight Deficit' THEN 0.33 ELSE 0.0 END AS balance_level,
        CASE growth_profile WHEN 'Strong Grower' THEN 1.0 WHEN 'Moderate Grower' THEN 0.67 WHEN 'Weak Grower' THEN 0.33 ELSE 0.0 END AS growth_level,
        CASE stability_profile WHEN 'Stable' THEN 1.0 WHEN 'Moderate' THEN 0.67 WHEN 'Volatile' THEN 0.33 ELSE 0.0 END AS stability_level,
        CASE consistency_profile WHEN 'Very Consistent' THEN 1.0 WHEN 'Consistent' THEN 0.67 WHEN 'Inconsistent' THEN 0.33 ELSE 0.0 END AS consistency_level
    FROM analysis.trade_structural
)
SELECT country_name, size_level, balance_level, growth_level, stability_level, consistency_level
FROM levels
WHERE country_name IN ('China', 'USA', 'Canada', 'Tunisia')
UNION ALL
SELECT 'World Average', AVG(size_level), AVG(balance_level), AVG(growth_level), AVG(stability_level), AVG(consistency_level)
FROM levels
ORDER BY country_name;

Running query in 'db'

,country_name,size_level,balance_level,growth_level,stability_level,consistency_level
0,Canada,1.00,0.33,0.33,0.67,0.33
1,China,1.00,1.00,0.33,0.67,1.00
2,Tunisia,0.00,0.00,0.33,0.67,0.33
3,USA,1.00,0.00,0.33,0.67,0.33
4,World Average,0.22,0.28,0.48,0.64,0.57


#### 3.7.5 Score Card

Fiche synthèse pour le pays choisi (`getvariable('country')`, même variable qu'en 3.1) : les 5 indicateurs (valeur brute, z-score vs monde, profil) plus le score global et son profil -- ex. `USA` a un score = X, un `balance_profile = Structural Deficit`, `size_profile = Leader`, `growth_profile = Weak Grower`, etc.

In [56]:
%%sql
WITH c AS (
    SELECT * FROM analysis.trade_structural WHERE country_name = getvariable('country')
)
SELECT 'Size Profile' AS indicator, export_share_structural AS value, z_export_share AS z_score, size_profile AS profile FROM c
UNION ALL
SELECT 'Balance Profile (Coverage)', coverage_ratio_structural, z_coverage_ratio, balance_profile FROM c
UNION ALL
SELECT 'Growth Profile', cagr_export, z_cagr_export, growth_profile FROM c
UNION ALL
SELECT 'Stability Profile', volatility_export, z_volatility_export, stability_profile FROM c
UNION ALL
SELECT 'Consistency Profile', negfreq_export, z_negfreq_export, consistency_profile FROM c
UNION ALL
SELECT 'Overall Score', structural_score, NULL, structural_profile FROM c;

Running query in 'db'

,indicator,value,z_score,profile
0,Size Profile,9.09,5.52,Leader
1,Balance Profile (Coverage),0.64,-0.09,Structural Deficit
2,Growth Profile,0.04,-0.21,Weak Grower
3,Stability Profile,10.97,-0.29,Moderate
4,Consistency Profile,0.40,0.47,Inconsistent
5,Overall Score,3.28,NaN,Strong


#### 3.7.6 Heatmap

Remplacé par des tables de classement : top 5 / bottom 5 par score, puis les pays au profil global `Strong` -- même lecture comparative que la heatmap, en tableau.

In [57]:
%%sql
SELECT * FROM (
    SELECT country_name, structural_score, structural_profile,
           size_profile, balance_profile, growth_profile, stability_profile, consistency_profile
    FROM analysis.trade_structural ORDER BY structural_score DESC LIMIT 5
) top5
UNION ALL
SELECT * FROM (
    SELECT country_name, structural_score, structural_profile,
           size_profile, balance_profile, growth_profile, stability_profile, consistency_profile
    FROM analysis.trade_structural ORDER BY structural_score ASC LIMIT 5
) bottom5
ORDER BY structural_score DESC;

Running query in 'db'

,country_name,structural_score,structural_profile,size_profile,balance_profile,growth_profile,stability_profile,consistency_profile
0,China,5.60,Strong,Leader,Strong Surplus,Weak Grower,Moderate,Very Consistent
1,USA,3.28,Strong,Leader,Structural Deficit,Weak Grower,Moderate,Inconsistent
2,Germany,2.97,Strong,Leader,Strong Surplus,Weak Grower,Moderate,Consistent
3,Japan,1.18,Strong,Leader,Slight Deficit,Weak Grower,Moderate,Inconsistent
4,Netherlands,1.03,Strong,Leader,Slight Surplus,Weak Grower,Moderate,Inconsistent
5,Sudan,-0.60,Weak,Small,Structural Deficit,Declining,Moderate,Inconsistent
6,Saint Lucia,-0.76,Weak,Small,Structural Deficit,Declining,Moderate,Very Inconsistent
7,Panama,-0.84,Weak,Small,Structural Deficit,Declining,Moderate,Very Inconsistent
8,Tonga,-0.91,Weak,Small,Structural Deficit,Declining,Very Volatile,Very Inconsistent
9,Yemen,-1.38,Weak,Small,Structural Deficit,Weak Grower,Very Volatile,Very Consistent


In [58]:
%%sql
SELECT country_name, structural_score,
       size_profile, balance_profile, growth_profile, stability_profile, consistency_profile
FROM analysis.trade_structural
WHERE structural_profile = 'Strong'
ORDER BY structural_score DESC;

Running query in 'db'

,country_name,structural_score,size_profile,balance_profile,growth_profile,stability_profile,consistency_profile
0,China,5.60,Leader,Strong Surplus,Weak Grower,Moderate,Very Consistent
1,USA,3.28,Leader,Structural Deficit,Weak Grower,Moderate,Inconsistent
2,Germany,2.97,Leader,Strong Surplus,Weak Grower,Moderate,Consistent
3,Japan,1.18,Leader,Slight Deficit,Weak Grower,Moderate,Inconsistent
4,Netherlands,1.03,Leader,Slight Surplus,Weak Grower,Moderate,Inconsistent
5,"China, Hong Kong SAR",0.98,Leader,Slight Deficit,Weak Grower,Moderate,Consistent
6,Italy,0.96,Leader,Slight Surplus,Weak Grower,Moderate,Consistent
7,Rep. of Korea,0.95,Leader,Slight Surplus,Weak Grower,Moderate,Inconsistent
8,France,0.92,Leader,Slight Deficit,Weak Grower,Moderate,Inconsistent
9,Mexico,0.82,Leader,Slight Deficit,Moderate Grower,Moderate,Very Consistent


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 4. Cross-Country Comparison
</div>

Même principe qu'en 3.1 : une variable native DuckDB, mais cette fois une **liste** (`SET VARIABLE countries = [...]`), lue via `UNNEST(getvariable('countries'))` dans un `IN (SELECT ...)`. Un seul endroit à modifier pour changer l'échantillon de pays comparés.

In [59]:
%%sql
SET VARIABLE countries = ['USA', 'China', 'France', 'Germany', 'Qatar', 'Saudi Arabia', 'Brazil', 'Tunisia'];

Running query in 'db'

,Success


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.1 Import / Export Comparison
</div>

In [60]:
%%sql
SELECT country_name, year, X, M
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,X,M
0,Brazil,2015,186.77B,180.46B
1,Brazil,2016,179.53B,145.25B
2,Brazil,2017,214.99B,165.86B
3,Brazil,2018,231.89B,192.84B
4,Brazil,2019,221.13B,193.16B
...,...,...,...,...
75,USA,2020,1.43T,2.41T
76,USA,2021,1.75T,2.93T
77,USA,2022,2.06T,3.37T
78,USA,2023,2.02T,3.17T


In [61]:
%%sql
SELECT country_name, X, M
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
  AND year = 2024
ORDER BY X DESC;

Running query in 'db'

,country_name,X,M
0,China,3.58T,2.59T
1,USA,2.06T,3.36T
2,Germany,1.63T,1.38T
3,France,645.96B,762.04B
4,Brazil,337.04B,277.55B
5,Saudi Arabia,305.50B,232.81B
6,Qatar,95.04B,35.80B
7,Tunisia,19.98B,26.07B


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.2 Trade Balance Comparison
</div>

In [62]:
%%sql
SELECT country_name, year, trade_balance
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,trade_balance
0,Brazil,2015,6.32B
1,Brazil,2016,34.28B
2,Brazil,2017,49.13B
3,Brazil,2018,39.05B
4,Brazil,2019,27.96B
...,...,...,...
75,USA,2020,-975.13B
76,USA,2021,-1.18T
77,USA,2022,-1.31T
78,USA,2023,-1.15T


In [63]:
%%sql
SELECT country_name, trade_balance
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
  AND year = 2024
ORDER BY trade_balance DESC;

Running query in 'db'

,country_name,trade_balance
0,China,991.41B
1,Germany,253.80B
2,Saudi Arabia,72.69B
3,Brazil,59.49B
4,Qatar,59.24B
5,Tunisia,-6.09B
6,France,-116.09B
7,USA,-1.29T


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.3 Share Comparison
</div>

In [64]:
%%sql
SELECT country_name, year, export_share, import_share
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,export_share,import_share
0,Brazil,2015,1.18,1.13
1,Brazil,2016,1.16,0.93
2,Brazil,2017,1.27,0.96
3,Brazil,2018,1.24,1.01
4,Brazil,2019,1.22,1.04
...,...,...,...,...
75,USA,2020,8.50,14.08
76,USA,2021,8.27,13.59
77,USA,2022,8.92,13.93
78,USA,2023,9.11,13.91


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.4 Growth Comparison
</div>

CAGR export déjà dans `analysis.trade_dynamic`. Indice base 100 (2015 = 100) calculé via `FIRST(X ORDER BY year ASC) OVER (PARTITION BY country_name)` -- équivalent SQL du `.div(df_base.loc[base_year]) * 100` pandas.

In [65]:
%%sql
SELECT country_name, cagr_export, cagr_import
FROM analysis.trade_dynamic
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY cagr_export DESC;

Running query in 'db'

,country_name,cagr_export,cagr_import
0,Brazil,0.07,0.05
1,China,0.05,0.05
2,Saudi Arabia,0.05,0.03
3,Tunisia,0.04,0.03
4,USA,0.04,0.04
5,France,0.03,0.03
6,Qatar,0.02,0.01
7,Germany,0.02,0.03


In [66]:
%%sql
SELECT
    country_name, year,
    X / FIRST(X) OVER (PARTITION BY country_name ORDER BY year ASC) * 100 AS export_index_base100
FROM analysis.trade_by_country_year
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,export_index_base100
0,Brazil,2015,100.00
1,Brazil,2016,96.12
2,Brazil,2017,115.11
3,Brazil,2018,124.15
4,Brazil,2019,118.39
...,...,...,...
75,USA,2020,95.23
76,USA,2021,116.73
77,USA,2022,137.30
78,USA,2023,134.40


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.5 Structural Vulnerability Profile
</div>

4 dimensions de risque (0 = faible, 1 = élevé), normalisées min-max sur l'ensemble des 183 pays de `analysis.trade_structural` -- `normalize_series` pandas équivaut à `(x - MIN(x) OVER()) / (MAX(x) OVER() - MIN(x) OVER())`. `balance_risk` plafonne `coverage_ratio_structural` au 95e centile avant normalisation (`LEAST(..., p95)`), pour éviter qu'un pays extrême n'écrase l'échelle. Colonnes ajoutées directement à `analysis.trade_structural` (DuckDB supporte `CREATE OR REPLACE TABLE t AS SELECT *, ... FROM t`).

In [67]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_structural AS
WITH capped AS (
    SELECT *,
        LEAST(coverage_ratio_structural, QUANTILE_CONT(coverage_ratio_structural, 0.95) OVER ()) AS coverage_ratio_capped
    FROM analysis.trade_structural
)
SELECT
    * EXCLUDE (coverage_ratio_capped),
    1 - (cagr_export - MIN(cagr_export) OVER ()) / (MAX(cagr_export) OVER () - MIN(cagr_export) OVER ())               AS growth_risk,
    (volatility_export - MIN(volatility_export) OVER ()) / (MAX(volatility_export) OVER () - MIN(volatility_export) OVER ()) AS stability_risk,
    (negfreq_export - MIN(negfreq_export) OVER ()) / (MAX(negfreq_export) OVER () - MIN(negfreq_export) OVER ())       AS consistency_risk,
    1 - (coverage_ratio_capped - MIN(coverage_ratio_capped) OVER ()) / (MAX(coverage_ratio_capped) OVER () - MIN(coverage_ratio_capped) OVER ()) AS balance_risk
FROM capped;

Running query in 'db'

,Success


In [68]:
%%sql
SELECT country_name, growth_risk, stability_risk, consistency_risk, balance_risk
FROM analysis.trade_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
UNION ALL
SELECT 'World Average', AVG(growth_risk), AVG(stability_risk), AVG(consistency_risk), AVG(balance_risk)
FROM analysis.trade_structural
ORDER BY country_name;

Running query in 'db'

,country_name,growth_risk,stability_risk,consistency_risk,balance_risk
0,Brazil,0.52,0.01,0.57,0.25
1,China,0.55,0.01,0.29,0.20
2,France,0.58,0.01,0.57,0.47
3,Germany,0.59,0.01,0.43,0.26
4,Qatar,0.59,0.04,0.71,0.00
5,Saudi Arabia,0.55,0.04,0.71,0.00
6,Tunisia,0.56,0.01,0.57,0.57
7,USA,0.57,0.01,0.57,0.61
8,World Average,0.55,0.03,0.48,0.56


In [69]:
%%sql
SELECT
    country_name, structural_score,
    growth_risk, stability_risk, consistency_risk, balance_risk,
    (growth_risk + stability_risk + consistency_risk + balance_risk) / 4 AS avg_risk
FROM analysis.trade_structural
ORDER BY avg_risk DESC
LIMIT 10;

Running query in 'db'

,country_name,structural_score,growth_risk,stability_risk,consistency_risk,balance_risk,avg_risk
0,Tonga,-0.91,0.88,0.21,0.82,0.92,0.71
1,Yemen,-1.38,0.56,1.00,0.29,0.96,0.70
2,Saint Lucia,-0.76,0.92,0.02,0.95,0.90,0.70
3,Panama,-0.84,1.00,0.03,1.00,0.70,0.68
4,"China, Macao SAR",-0.44,0.67,0.02,0.86,0.96,0.63
5,Mauritius,-0.49,0.69,0.01,1.00,0.80,0.62
6,Cabo Verde,-0.58,0.63,0.20,0.71,0.95,0.62
7,Samoa,-0.47,0.70,0.03,0.79,0.93,0.61
8,Curacao,-0.51,0.76,0.01,0.71,0.96,0.61
9,Maldives,-0.46,0.71,0.02,0.71,0.98,0.61


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.6 Positioning
</div>

Quadrants : positionnement relatif (visuel en pandas), classification par seuils absolus (ici).

- **Static** : export_share vs coverage_ratio → taille + balance
- **Dynamic** : cagr_export vs volatility_export → croissance + risque
- **Strategic** : export_share vs cagr_export → positionnement type BCG
- **Profile** : synthèse `meta_profile` combinant les trois

**Écart avec pandas à noter** : le notebook pandas calcule ces classifications **deux fois** -- une première fois (cellules 4.6.1) avec des seuils **médians** et une fonction `compute_meta_profile` qui contient une faute de frappe (`strategic == 'Stars'` au lieu de `'Star'`, qui ne matche donc jamais), puis une seconde fois (cellules 4.6.5) avec des **seuils fixes** (1.0) et la fonction corrigée -- cette seconde version **écrase** la première dans `df_structural`. C'est elle qui est réellement sauvegardée (`df_structural.to_csv`) et donc la seule répliquée ici -- vérifié : les comptages `meta_profile` (17/11/58/73/24) correspondent exactement au tableau documenté dans le notebook pandas.

#### 4.6.1 Quadrant Classification

In [70]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_structural AS
WITH quadrants AS (
    SELECT
        *,
        CASE
            WHEN export_share_structural >= 1.0 AND coverage_ratio_structural >= 1.0 THEN 'Dominant'
            WHEN export_share_structural >= 1.0 AND coverage_ratio_structural <  1.0 THEN 'Vulnerable'
            WHEN export_share_structural <  1.0 AND coverage_ratio_structural >= 1.0 THEN 'Balanced'
            ELSE 'Marginal'
        END AS static_quadrant,
        CASE
            WHEN cagr_export >= 0 AND volatility_export <= MEDIAN(volatility_export) OVER () THEN 'Star'
            WHEN cagr_export >= 0 AND volatility_export >  MEDIAN(volatility_export) OVER () THEN 'Risky Grower'
            WHEN cagr_export <  0 AND volatility_export <= MEDIAN(volatility_export) OVER () THEN 'Stable Stagnant'
            ELSE 'Declining'
        END AS dynamic_quadrant,
        CASE
            WHEN export_share_structural >= 1.0 AND cagr_export >= 0 THEN 'Star'
            WHEN export_share_structural >= 1.0 AND cagr_export <  0 THEN 'Cash Cow'
            WHEN export_share_structural <  1.0 AND cagr_export >= 0 THEN 'Emerging'
            ELSE 'Marginal'
        END AS strategic_quadrant
    FROM analysis.trade_structural
)
SELECT
    *,
    static_quadrant || ' | ' || dynamic_quadrant || ' | ' || strategic_quadrant AS trade_profile,
    CASE
        WHEN static_quadrant = 'Dominant' AND strategic_quadrant = 'Star' THEN 'Global Leader'
        WHEN static_quadrant IN ('Dominant', 'Vulnerable') AND strategic_quadrant IN ('Star', 'Cash Cow') THEN 'Established Power'
        WHEN strategic_quadrant = 'Emerging' AND dynamic_quadrant = 'Star' THEN 'Rising Challenger'
        WHEN strategic_quadrant = 'Emerging' AND dynamic_quadrant = 'Risky Grower' THEN 'Volatile Emerging'
        WHEN dynamic_quadrant = 'Declining' OR (strategic_quadrant = 'Marginal' AND dynamic_quadrant = 'Stable Stagnant') THEN 'Peripheral'
        ELSE 'Transitional'
    END AS meta_profile
FROM quadrants;

Running query in 'db'

,Success


In [71]:
%%sql
WITH unpivoted AS (
    UNPIVOT (
        SELECT static_quadrant, dynamic_quadrant, strategic_quadrant, meta_profile
        FROM analysis.trade_structural
    )
    ON static_quadrant, dynamic_quadrant, strategic_quadrant, meta_profile
    INTO NAME dimension VALUE category
)
SELECT dimension, category, COUNT(*) AS n_countries
FROM unpivoted
GROUP BY dimension, category
ORDER BY dimension, n_countries DESC;

Running query in 'db'

,dimension,category,n_countries
0,dynamic_quadrant,Star,84
1,dynamic_quadrant,Risky Grower,75
2,dynamic_quadrant,Declining,16
3,dynamic_quadrant,Stable Stagnant,8
4,meta_profile,Volatile Emerging,73
5,meta_profile,Rising Challenger,58
6,meta_profile,Peripheral,24
7,meta_profile,Global Leader,17
8,meta_profile,Established Power,11
9,static_quadrant,Marginal,121


#### 4.6.2 Structural Positioning

Pas de nouveau calcul -- en pandas c'est la trajectoire animée (export_share vs coverage_ratio par année) des pays de `countries`, déjà couverte par 4.1/4.3 (mêmes séries). Seule info nouvelle : le `static_quadrant` de chacun.

In [72]:
%%sql
SELECT country_name, export_share_structural, coverage_ratio_structural, static_quadrant
FROM analysis.trade_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY export_share_structural DESC;

Running query in 'db'

,country_name,export_share_structural,coverage_ratio_structural,static_quadrant
0,China,14.79,1.29,Dominant
1,USA,9.09,0.64,Vulnerable
2,Germany,8.24,1.19,Dominant
3,France,3.03,0.85,Vulnerable
4,Saudi Arabia,1.36,1.60,Dominant
5,Brazil,1.25,1.21,Dominant
6,Qatar,0.43,2.58,Balanced
7,Tunisia,0.08,0.70,Marginal


#### 4.6.3 Dynamic Positioning

`resilience_export` n'est pas stockée dans `trade_structural` (seuls ses 3 composants z-score le sont) -- recalculée à la volée avec la même formule qu'en F.1.2 (`z_cagr - z_volatility - z_negfreq`).

In [73]:
%%sql
SELECT
    country_name, cagr_export, volatility_export,
    z_cagr_export - z_volatility_export - z_negfreq_export AS resilience_export,
    dynamic_quadrant
FROM analysis.trade_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY resilience_export DESC;

Running query in 'db'

,country_name,cagr_export,volatility_export,resilience_export,dynamic_quadrant
0,China,0.05,10.47,1.22,Star
1,Germany,0.02,8.04,0.20,Star
2,Brazil,0.07,13.91,-0.04,Star
3,Tunisia,0.04,8.85,-0.32,Star
4,USA,0.04,10.97,-0.39,Star
5,France,0.03,8.44,-0.43,Star
6,Saudi Arabia,0.05,32.60,-1.30,Risky Grower
7,Qatar,0.02,35.70,-1.62,Risky Grower


#### 4.6.4 Strategic Positioning

In [74]:
%%sql
SELECT country_name, export_share_structural, cagr_export, structural_score, strategic_quadrant
FROM analysis.trade_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY structural_score DESC;

Running query in 'db'

,country_name,export_share_structural,cagr_export,structural_score,strategic_quadrant
0,China,14.79,0.05,5.60,Star
1,USA,9.09,0.04,3.28,Star
2,Germany,8.24,0.02,2.97,Star
3,France,3.03,0.03,0.92,Star
4,Brazil,1.25,0.07,0.30,Star
5,Saudi Arabia,1.36,0.05,0.25,Star
6,Qatar,0.43,0.02,-0.15,Emerging
7,Tunisia,0.08,0.04,-0.20,Emerging


#### 4.6.5 Profile Positioning

Distribution des `meta_profile` déjà montrée en 4.6.1. Ici, la vue synthétique par pays : les 3 quadrants + le `meta_profile` final pour l'échantillon `countries`.

In [75]:
%%sql
SELECT country_name, structural_score, static_quadrant, dynamic_quadrant, strategic_quadrant, meta_profile
FROM analysis.trade_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY structural_score DESC;

Running query in 'db'

,country_name,structural_score,static_quadrant,dynamic_quadrant,strategic_quadrant,meta_profile
0,China,5.60,Dominant,Star,Star,Global Leader
1,USA,3.28,Vulnerable,Star,Star,Established Power
2,Germany,2.97,Dominant,Star,Star,Global Leader
3,France,0.92,Vulnerable,Star,Star,Established Power
4,Brazil,0.30,Dominant,Star,Star,Global Leader
5,Saudi Arabia,0.25,Dominant,Risky Grower,Star,Global Leader
6,Qatar,-0.15,Balanced,Risky Grower,Emerging,Volatile Emerging
7,Tunisia,-0.20,Marginal,Star,Emerging,Rising Challenger


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 4.7 Save Table
</div>

Rien à coder -- `analysis.trade_structural` est déjà une table persistée dans `data/sql/trade_analysis_sql.duckdb` (équivalent du `df_structural.to_csv(...)` pandas), à jour avec toutes les colonnes ajoutées en 3.7, 4.5 et 4.6.

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
# G. Bilateral Trade Analysis
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 1. Objectives, Analytical Framework and Structure
</div>

This section analyzes trade at the bilateral level — not global aggregates.
Three dimensions are covered : partner concentration structure (5), asymmetric
dependency between country pairs (6–7), most critical dependencies (8),
and bilateral typology profiles over time (9–10).
The central concept is the **Bilateral Dependency Index (BDI)** — measuring
not just trade volume but the asymmetry of dependence between trading partners.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 2. Methodology and Theoretical Definitions
</div>

**HHI (Herfindahl-Hirschman Index)** — measures partner concentration.
HHI = Σ(share_i²) × 10,000. Range 0–10,000. High HHI = few dominant partners.

**BDI (Bilateral Dependency Index)** — measures asymmetric dependence
between a country pair (A, B) : BDI = share of B in A's trade − share of A
in B's trade. Positive = A depends more on B than B on A.

**BDIS (Structural BDI)** — full-period aggregation of BDI,
capturing structural rather than annual dependency.

**Bilateral Profile** — composite classification combining HHI level,
BDI asymmetry, and dependency score into a typology
(Robust / Competitive / Vulnerable / Fragile).

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 3. Construction of Bilateral Analytical Tables
</div>

Base = `analysis.trade` (jointure de E.2), filtrée sur `cmdCode='TOTAL'`, `typeCode='C'`, `flow_code IN ('M','X')` -- comme en F.1.1, mais **sans** agréger sur les partenaires. Deux tables : `analysis.trade_world` (partner_name='World', mêmes lignes que la base de F.1.1 avant pivot -- réutilisée plus loin en G.9 pour rattacher le volume total) et `analysis.trade_bilateral` (partner_name != 'World', le détail par partenaire, avec `total`/`share` calculés par `SUM(...) OVER (PARTITION BY country_name, year, flow_code)` -- équivalent SQL du `.transform('sum')` pandas, pas besoin de self-join).

In [76]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_world AS
SELECT country_code, country_iso3, country_name, flow_code, year, value
FROM analysis.trade
WHERE cmdCode = 'TOTAL'
  AND partner_name = 'World'
  AND flow_code IN ('M', 'X')
  AND typeCode = 'C';

Running query in 'db'

,Success


In [77]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bilateral AS
SELECT
    country_name, country_iso3, partner_name, partner_iso3, year, flow_code, value,
    SUM(value) OVER (PARTITION BY country_name, year, flow_code) AS total,
    value / SUM(value) OVER (PARTITION BY country_name, year, flow_code) AS share
FROM analysis.trade
WHERE cmdCode = 'TOTAL'
  AND partner_name != 'World'
  AND flow_code IN ('M', 'X')
  AND typeCode = 'C';

Running query in 'db'

,Success


In [78]:
%%sql
SELECT * FROM analysis.trade_bilateral USING SAMPLE 3;

Running query in 'db'

,country_name,country_iso3,partner_name,partner_iso3,year,flow_code,value,total,share
0,Dominican Rep.,DOM,Bolivia (Plurinational State of),BOL,2015,M,2.7M,18.69B,0.00
1,Djibouti,DJI,Denmark,DNK,2023,X,28.14,4.78B,0.00
2,Nigeria,NGA,Russian Federation,RUS,2018,M,1.02B,43.01B,0.02


In [79]:
%%sql
SELECT
    (SELECT COUNT(*) FROM analysis.trade_bilateral) AS n_rows_bilateral,
    (SELECT COUNT(DISTINCT country_name) FROM analysis.trade_bilateral) AS n_countries_bilateral,
    (SELECT COUNT(*) FROM analysis.trade_world) AS n_rows_world;

Running query in 'db'

,n_rows_bilateral,n_countries_bilateral,n_rows_world
0,540523,188,3475


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 4. Global Trade Flow Visualization
#### Sankey Diagram
</div>

Remplacé par une table : les 10 plus gros flux bilatéraux (tous pays confondus, pas seulement l'échantillon `countries` utilisé pour lisibilité dans le diagramme Sankey pandas), année 2024 -- reprend le tableau `top10` déjà imprimé en texte par la fonction pandas elle-même.

In [80]:
%%sql
SELECT country_name, partner_name, flow_code, value, share
FROM analysis.trade_bilateral
WHERE year = 2024
ORDER BY value DESC
LIMIT 10;

Running query in 'db'

,country_name,partner_name,flow_code,value,share
0,China,USA,X,525.65B,0.15
1,USA,Mexico,M,509.96B,0.15
2,Mexico,USA,X,503.42B,0.81
3,USA,China,M,462.62B,0.14
4,USA,Canada,M,421.21B,0.13
5,Canada,USA,X,419.61B,0.76
6,"China, Hong Kong SAR",China,X,377.80B,0.59
7,USA,Canada,X,348.41B,0.17
8,USA,Mexico,X,334.04B,0.16
9,China,"China, Hong Kong SAR",X,290.87B,0.08


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 5. Partners Structure & Bilateral Concentration
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.1 Partners Share
</div>

Réutilise la variable `country` (3.1). Snapshot 2024 (top 10 partenaires par part) pour export et import, puis évolution du top 5 export sur toute la période.

In [81]:
%%sql
SELECT partner_name, share
FROM analysis.trade_bilateral
WHERE country_name = getvariable('country') AND flow_code = 'X' AND year = 2024
ORDER BY share DESC
LIMIT 10;

Running query in 'db'

,partner_name,share
0,Canada,0.17
1,Mexico,0.16
2,China,0.07
3,Netherlands,0.04
4,United Kingdom,0.04
5,Japan,0.04
6,Germany,0.04
7,Rep. of Korea,0.03
8,Brazil,0.02
9,Singapore,0.02


In [82]:
%%sql
SELECT partner_name, share
FROM analysis.trade_bilateral
WHERE country_name = getvariable('country') AND flow_code = 'M' AND year = 2024
ORDER BY share DESC
LIMIT 10;

Running query in 'db'

,partner_name,share
0,Mexico,0.15
1,China,0.14
2,Canada,0.13
3,Germany,0.05
4,Japan,0.05
5,Viet Nam,0.04
6,Rep. of Korea,0.04
7,"Other Asia, nes",0.04
8,Ireland,0.03
9,India,0.03


In [83]:
%%sql
WITH top_partners AS (
    SELECT partner_name
    FROM analysis.trade_bilateral
    WHERE country_name = getvariable('country') AND flow_code = 'X' AND year = 2024
    ORDER BY share DESC
    LIMIT 5
)
SELECT b.partner_name, b.year, b.share
FROM analysis.trade_bilateral b
JOIN top_partners t ON b.partner_name = t.partner_name
WHERE b.country_name = getvariable('country') AND b.flow_code = 'X'
ORDER BY b.partner_name, b.year;

Running query in 'db'

,partner_name,year,share
0,Canada,2015,0.19
1,Canada,2016,0.18
2,Canada,2017,0.18
3,Canada,2018,0.18
4,Canada,2019,0.18
5,Canada,2020,0.18
6,Canada,2021,0.18
7,Canada,2022,0.17
8,Canada,2023,0.17
9,Canada,2024,0.17


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.2 Partners Concentration (TOP 3/5/10)
</div>

Part cumulée des top 3/5/10 partenaires par année -- même pattern qu'en F.2.5.1 (`ROW_NUMBER` + `FILTER`), appliqué ici par pays plutôt qu'au commerce mondial.

In [84]:
%%sql
WITH ranked AS (
    SELECT country_name, year, flow_code, share,
           ROW_NUMBER() OVER (PARTITION BY country_name, year, flow_code ORDER BY share DESC) AS rnk
    FROM analysis.trade_bilateral
)
SELECT year,
       SUM(share) FILTER (WHERE rnk <= 3)  AS top3_share,
       SUM(share) FILTER (WHERE rnk <= 5)  AS top5_share,
       SUM(share) FILTER (WHERE rnk <= 10) AS top10_share
FROM ranked
WHERE country_name = getvariable('country') AND flow_code = 'X'
GROUP BY year
ORDER BY year;

Running query in 'db'

,year,top3_share,top5_share,top10_share
0,2015,0.42,0.50,0.64
1,2016,0.42,0.50,0.64
2,2017,0.42,0.50,0.65
3,2018,0.41,0.50,0.64
4,2019,0.40,0.49,0.64
5,2020,0.41,0.50,0.65
6,2021,0.42,0.50,0.65
7,2022,0.40,0.48,0.63
8,2023,0.41,0.49,0.64
9,2024,0.40,0.48,0.64


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.3 Create HHI Dataframe
</div>

`hhi = SUM(share²)` par (pays, année, flux) -- `share` est déjà une fraction (0-1), donc `hhi` aussi, cohérent avec les seuils `classify_hhi_level` (0.08/0.15/0.25). `delta_hhi_norm` réplique `normalize_delta` pandas : normalisation min-max des `|delta_hhi|` **par flux** (`PARTITION BY flow_code`, sur tous pays/années confondus -- pas par pays), signe réappliqué, puis recentrée sur 0.5.

In [85]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bi_hhi AS
WITH agg AS (
    SELECT country_name, year, flow_code,
           SUM(share * share) AS hhi,
           COUNT(DISTINCT partner_name) AS n_partners
    FROM analysis.trade_bilateral
    GROUP BY country_name, year, flow_code
),
deltas AS (
    SELECT *,
        hhi - LAG(hhi) OVER (PARTITION BY country_name, flow_code ORDER BY year) AS delta_hhi
    FROM agg
),
normed AS (
    SELECT *,
        (hhi - 1.0 / n_partners) / (1 - 1.0 / n_partners) AS hhi_norm,
        COALESCE(delta_hhi, 0) AS delta_hhi_filled
    FROM deltas
),
bounds AS (
    SELECT *,
        MIN(ABS(delta_hhi_filled)) OVER (PARTITION BY flow_code) AS min_abs_delta,
        MAX(ABS(delta_hhi_filled)) OVER (PARTITION BY flow_code) AS max_abs_delta
    FROM normed
)
SELECT
    country_name, year, flow_code, n_partners, hhi, delta_hhi, hhi_norm,
    CASE
        WHEN max_abs_delta = 0 THEN 0.5
        ELSE ((ABS(delta_hhi_filled) - min_abs_delta) / (max_abs_delta - min_abs_delta) * SIGN(delta_hhi_filled) + 1) / 2
    END AS delta_hhi_norm,
    CASE
        WHEN hhi < 0.08 THEN 'Low'
        WHEN hhi < 0.15 THEN 'Moderate'
        WHEN hhi < 0.25 THEN 'High'
        ELSE 'Extreme'
    END AS hhi_level,
    CASE
        WHEN delta_hhi IS NULL OR delta_hhi = 0 THEN 'Stable'
        WHEN delta_hhi > 0.025 THEN 'Increasing'
        WHEN delta_hhi < -0.025 THEN 'Decreasing'
        ELSE 'Stable'
    END AS hhi_trend
FROM bounds
ORDER BY country_name, flow_code, year;

Running query in 'db'

,Success


In [86]:
%%sql
SELECT * FROM analysis.trade_bi_hhi USING SAMPLE 3;

Running query in 'db'

,country_name,year,flow_code,n_partners,hhi,delta_hhi,hhi_norm,delta_hhi_norm,hhi_level,hhi_trend
0,France,2016,X,230,0.06,0.00,0.05,0.50,Low,Stable
1,Grenada,2021,X,49,0.36,0.08,0.35,0.55,Extreme,Increasing
2,Paraguay,2025,M,175,0.18,0.01,0.18,0.51,High,Stable


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.4 HHI Time Evolution
</div>

In [87]:
%%sql
SELECT year, flow_code, hhi, delta_hhi, hhi_level, hhi_trend
FROM analysis.trade_bi_hhi
WHERE country_name = getvariable('country')
ORDER BY flow_code, year;

Running query in 'db'

,year,flow_code,hhi,delta_hhi,hhi_level,hhi_trend
0,2015,M,0.09,NaN,Moderate,Stable
1,2016,M,0.09,-0.00,Moderate,Stable
2,2017,M,0.09,0.00,Moderate,Stable
3,2018,M,0.09,-0.00,Moderate,Stable
4,2019,M,0.08,-0.01,Moderate,Stable
5,2020,M,0.08,-0.00,Moderate,Stable
6,2021,M,0.08,-0.00,Low,Stable
7,2022,M,0.08,-0.00,Low,Stable
8,2023,M,0.07,-0.00,Low,Stable
9,2024,M,0.07,-0.00,Low,Stable


In [88]:
%%sql
SELECT flow_code, year AS peak_year, hhi AS peak_hhi
FROM analysis.trade_bi_hhi
WHERE country_name = getvariable('country')
QUALIFY ROW_NUMBER() OVER (PARTITION BY flow_code ORDER BY hhi DESC) = 1;

Running query in 'db'

,flow_code,peak_year,peak_hhi
0,M,2015,0.09
1,X,2016,0.08


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.5 Dependency (Import score vs Export score)
</div>

Réutilise la liste `countries` (section 4). Seuil de diversification HHI = 0.10, mêmes 4 quadrants qu'en pandas.

In [89]:
%%sql
WITH exp AS (
    SELECT country_name, hhi AS hhi_export FROM analysis.trade_bi_hhi
    WHERE flow_code = 'X' AND year = 2024 AND country_name IN (SELECT UNNEST(getvariable('countries')))
),
imp AS (
    SELECT country_name, hhi AS hhi_import FROM analysis.trade_bi_hhi
    WHERE flow_code = 'M' AND year = 2024 AND country_name IN (SELECT UNNEST(getvariable('countries')))
)
SELECT
    e.country_name, e.hhi_export, i.hhi_import,
    CASE
        WHEN e.hhi_export >= 0.10 AND i.hhi_import >= 0.10 THEN 'Dual Exposed'
        WHEN e.hhi_export >= 0.10 AND i.hhi_import <  0.10 THEN 'Export Exposed'
        WHEN e.hhi_export <  0.10 AND i.hhi_import >= 0.10 THEN 'Import Exposed'
        ELSE 'Diversified'
    END AS quadrant
FROM exp e JOIN imp i ON e.country_name = i.country_name
ORDER BY e.country_name;

Running query in 'db'

,country_name,hhi_export,hhi_import,quadrant
0,Brazil,0.10,0.10,Export Exposed
1,China,0.04,0.04,Diversified
2,France,0.05,0.05,Diversified
3,Germany,0.04,0.05,Diversified
4,Qatar,0.09,0.06,Diversified
5,Saudi Arabia,0.54,0.08,Export Exposed
6,Tunisia,0.11,0.06,Export Exposed
7,USA,0.07,0.07,Diversified


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.6 HHI Internal Structure : Contribution of partners to HHI
</div>

Pareto : contribution de chaque partenaire au HHI total (`share²`), triée décroissante, avec part cumulée -- révèle si la concentration vient d'un seul partenaire dominant ou d'une diversité de partenaires moyens.

In [90]:
%%sql
WITH contrib AS (
    SELECT partner_name, share * share AS hhi_contribution
    FROM analysis.trade_bilateral
    WHERE country_name = getvariable('country') AND flow_code = 'X' AND year = 2024
),
totaled AS (
    SELECT *, SUM(hhi_contribution) OVER () AS total_hhi FROM contrib
)
SELECT
    partner_name,
    hhi_contribution / total_hhi * 100 AS hhi_pct,
    SUM(hhi_contribution) OVER (ORDER BY hhi_contribution DESC) / total_hhi * 100 AS cumulative_pct
FROM totaled
ORDER BY hhi_contribution DESC
LIMIT 15;

Running query in 'db'

,partner_name,hhi_pct,cumulative_pct
0,Canada,39.80,39.80
1,Mexico,36.58,76.38
2,China,6.76,83.14
3,Netherlands,2.63,85.77
4,United Kingdom,2.09,87.86
5,Japan,2.08,89.95
6,Germany,1.86,91.81
7,Rep. of Korea,1.41,93.22
8,Brazil,0.81,94.03
9,Singapore,0.69,94.72


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 5.7 Breaks Detection
</div>

Chocs de concentration : années où `|delta_hhi| >= 0.01`, pour `country`.

In [91]:
%%sql
SELECT year, flow_code, delta_hhi,
       CASE WHEN delta_hhi > 0 THEN 'Concentration UP' ELSE 'Diversification DOWN' END AS direction
FROM analysis.trade_bi_hhi
WHERE country_name = getvariable('country') AND ABS(delta_hhi) >= 0.01
ORDER BY flow_code, year;

Running query in 'db'

,year,flow_code,delta_hhi,direction


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 6. Asymmetric Bilateral Dependency
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 6.1 Create DBI TABLE
</div>

`dependency` = part du partenaire dans le commerce total du pays (import+export combinés, contrairement à `trade_bilateral.share` qui est par flux). `bdi` = auto-jointure de `dep` sur elle-même en inversant (country, partner) -- équivalent SQL du self-merge pandas -- puis `|dependency - dependency_reverse|`.

In [92]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bdi AS
WITH bilateral_total AS (
    SELECT country_name, partner_name, year, SUM(value) AS bilateral_trade
    FROM analysis.trade_bilateral
    GROUP BY country_name, partner_name, year
),
country_total AS (
    SELECT country_name, year, SUM(value) AS total_trade
    FROM analysis.trade_bilateral
    GROUP BY country_name, year
),
dep AS (
    SELECT b.country_name, b.partner_name, b.year,
           b.bilateral_trade / c.total_trade AS dependency
    FROM bilateral_total b
    JOIN country_total c ON b.country_name = c.country_name AND b.year = c.year
)
SELECT
    d.country_name, d.partner_name, d.year,
    d.dependency,
    r.dependency AS dependency_reverse,
    ABS(d.dependency - r.dependency) AS bdi,
    LEAST(ABS(d.dependency - r.dependency), 1.0) AS bdi_norm,
    CASE WHEN d.dependency < 0.15 THEN 'Low' WHEN d.dependency < 0.25 THEN 'Moderate' WHEN d.dependency < 0.45 THEN 'High' ELSE 'Critical' END AS dep_level,
    CASE WHEN r.dependency < 0.15 THEN 'Low' WHEN r.dependency < 0.25 THEN 'Moderate' WHEN r.dependency < 0.45 THEN 'High' ELSE 'Critical' END AS dep_reverse_level,
    CASE
        WHEN ABS(d.dependency - r.dependency) < 0.2 THEN 'Interdependence'
        WHEN ABS(d.dependency - r.dependency) < 0.5 THEN 'Moderate Dependance'
        WHEN ABS(d.dependency - r.dependency) < 0.8 THEN 'High Dependance'
        ELSE 'Critical Dependance'
    END AS bdi_level,
    CASE WHEN d.dependency > r.dependency THEN d.partner_name ELSE d.country_name END AS dominant_partner,
    CASE WHEN d.dependency > r.dependency THEN d.country_name || ' depends on ' || d.partner_name
         ELSE d.partner_name || ' depends on ' || d.country_name END AS dependency_type
FROM dep d
JOIN dep r ON d.country_name = r.partner_name AND d.partner_name = r.country_name AND d.year = r.year;

Running query in 'db'

,Success


In [93]:
%%sql
SELECT * FROM analysis.trade_bdi USING SAMPLE 3;

Running query in 'db'

,country_name,partner_name,year,dependency,dependency_reverse,bdi,bdi_norm,dep_level,dep_reverse_level,bdi_level,dominant_partner,dependency_type
0,Mauritius,Rwanda,2021,0.00,0.00,0.00,0.00,Low,Low,Interdependence,Mauritius,Rwanda depends on Mauritius
1,Egypt,Denmark,2020,0.00,0.00,0.00,0.00,Low,Low,Interdependence,Denmark,Egypt depends on Denmark
2,Malaysia,Italy,2025,0.01,0.00,0.00,0.00,Low,Low,Interdependence,Italy,Malaysia depends on Italy


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 6.2 BDI for a pair of countries
</div>

In [94]:
%%sql
SELECT year, dependency, dependency_reverse, bdi, dominant_partner
FROM analysis.trade_bdi
WHERE country_name = 'Canada' AND partner_name = 'USA'
ORDER BY year;

Running query in 'db'

,year,dependency,dependency_reverse,bdi,dominant_partner
0,2015,0.65,0.15,0.50,USA
1,2016,0.64,0.15,0.49,USA
2,2017,0.63,0.15,0.49,USA
3,2018,0.63,0.15,0.48,USA
4,2019,0.63,0.15,0.48,USA
5,2020,0.61,0.14,0.47,USA
6,2021,0.62,0.14,0.48,USA
7,2022,0.63,0.15,0.48,USA
8,2023,0.64,0.15,0.48,USA
9,2024,0.63,0.14,0.49,USA


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 6.3 Dependency Cartography (Top parteners)
</div>

Objectif : identifier les partenaires critiques pour le pays choisi. Les 10 principaux partenaires sont sélectionnés par part **export** (resp. **import**), mais le `bdi`/`dependency` affiché reste toujours la version combinée import+export de `trade_bdi` (comme en pandas).

In [95]:
%%sql
WITH top_partners AS (
    SELECT partner_name
    FROM analysis.trade_bilateral
    WHERE country_name = getvariable('country') AND flow_code = 'X' AND year = 2024
    ORDER BY share DESC
    LIMIT 10
)
SELECT b.partner_name, b.dependency, b.dependency_reverse, b.bdi, b.bdi_level, b.dominant_partner
FROM analysis.trade_bdi b
JOIN top_partners t ON b.partner_name = t.partner_name
WHERE b.country_name = getvariable('country') AND b.year = 2024
ORDER BY b.dependency DESC;

Running query in 'db'

,partner_name,dependency,dependency_reverse,bdi,bdi_level,dominant_partner
0,Mexico,0.16,0.61,0.45,Moderate Dependance,USA
1,Canada,0.14,0.63,0.49,Moderate Dependance,USA
2,China,0.11,0.11,0.00,Interdependence,USA
3,Germany,0.04,0.09,0.04,Interdependence,USA
4,Japan,0.04,0.16,0.11,Interdependence,USA
5,Rep. of Korea,0.04,0.15,0.12,Interdependence,USA
6,United Kingdom,0.03,0.13,0.10,Interdependence,USA
7,Netherlands,0.02,0.08,0.05,Interdependence,USA
8,Brazil,0.02,0.14,0.12,Interdependence,USA
9,Singapore,0.02,0.10,0.09,Interdependence,USA


In [96]:
%%sql
WITH top_partners AS (
    SELECT partner_name
    FROM analysis.trade_bilateral
    WHERE country_name = getvariable('country') AND flow_code = 'M' AND year = 2024
    ORDER BY share DESC
    LIMIT 10
)
SELECT b.partner_name, b.dependency, b.dependency_reverse, b.bdi, b.bdi_level, b.dominant_partner
FROM analysis.trade_bdi b
JOIN top_partners t ON b.partner_name = t.partner_name
WHERE b.country_name = getvariable('country') AND b.year = 2024
ORDER BY b.dependency DESC;

Running query in 'db'

,partner_name,dependency,dependency_reverse,bdi,bdi_level,dominant_partner
0,Mexico,0.16,0.61,0.45,Moderate Dependance,USA
1,Canada,0.14,0.63,0.49,Moderate Dependance,USA
2,China,0.11,0.11,0.00,Interdependence,USA
3,Germany,0.04,0.09,0.04,Interdependence,USA
4,Japan,0.04,0.16,0.11,Interdependence,USA
5,Rep. of Korea,0.04,0.15,0.12,Interdependence,USA
6,India,0.02,0.10,0.08,Interdependence,USA
7,Ireland,0.02,0.27,0.24,Moderate Dependance,USA


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 6.4 Heatmap BDI
</div>

Remplacé par une table (au lieu d'une matrice colorée) : tous les couples pays/partenaire au sein de `countries`, année 2024, triés par BDI décroissant -- la paire la plus asymétrique apparaît en premier.

In [97]:
%%sql
SELECT country_name, partner_name, dependency, dependency_reverse, bdi, bdi_level, dominant_partner
FROM analysis.trade_bdi
WHERE year = 2024
  AND country_name IN (SELECT UNNEST(getvariable('countries')))
  AND partner_name IN (SELECT UNNEST(getvariable('countries')))
  AND country_name != partner_name
ORDER BY bdi DESC;

Running query in 'db'

,country_name,partner_name,dependency,dependency_reverse,bdi,bdi_level,dominant_partner
0,China,Brazil,0.03,0.27,0.24,Moderate Dependance,China
1,Brazil,China,0.27,0.03,0.24,Moderate Dependance,China
2,China,Qatar,0.00,0.19,0.18,Interdependence,China
3,Qatar,China,0.19,0.00,0.18,Interdependence,China
4,Tunisia,France,0.15,0.01,0.15,Interdependence,France
5,France,Tunisia,0.01,0.15,0.15,Interdependence,France
6,USA,Brazil,0.02,0.14,0.12,Interdependence,USA
7,Brazil,USA,0.14,0.02,0.12,Interdependence,USA
8,China,Saudi Arabia,0.02,0.12,0.10,Interdependence,China
9,Saudi Arabia,China,0.12,0.02,0.10,Interdependence,China


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 7. Structural Bilateral Dependency Index (BDI)
</div>

BDI vs Structural BDI

BDI measures asymmetry between two specific countries — who depends more on whom in a bilateral relationship.
Structural BDI measures the internal export/import imbalance of a single country across all its partners — how differently its trade is distributed between exports and imports globally.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 7.1 Create DBI STRUCTURAL TABLE
</div>

`bdis = SUM(|X_share - M_share|) / 2` par pays/année, sur tous les partenaires (pivot X/M des parts). **Écart de granularité à noter** : contrairement à `delta_hhi_norm` en 5.3 (normalisé **par flux**), `delta_bdis_norm` ici réplique `normalize_delta(df_bdi_struct['delta_bdis'])` appelé **directement sur toute la colonne**, sans `groupby` -- donc un seul MIN/MAX global (`OVER ()` sans `PARTITION BY`), pas par pays ni par flux.

In [98]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bdi_structural AS
WITH pivoted AS (
    SELECT country_name, year, partner_name,
           SUM(share) FILTER (WHERE flow_code = 'X') AS x_share,
           SUM(share) FILTER (WHERE flow_code = 'M') AS m_share
    FROM analysis.trade_bilateral
    GROUP BY country_name, year, partner_name
),
abs_diff AS (
    SELECT country_name, year,
           SUM(ABS(COALESCE(x_share, 0) - COALESCE(m_share, 0))) AS total_abs_diff
    FROM pivoted
    GROUP BY country_name, year
),
partners AS (
    SELECT country_name, year, COUNT(DISTINCT partner_name) AS n_partners
    FROM analysis.trade_bilateral
    GROUP BY country_name, year
),
base AS (
    SELECT a.country_name, a.year, p.n_partners, a.total_abs_diff / 2 AS bdis
    FROM abs_diff a
    JOIN partners p ON a.country_name = p.country_name AND a.year = p.year
),
deltas AS (
    SELECT *,
        bdis - LAG(bdis) OVER (PARTITION BY country_name ORDER BY year) AS delta_bdis,
        COALESCE(bdis - LAG(bdis) OVER (PARTITION BY country_name ORDER BY year), 0) AS delta_filled
    FROM base
),
bounds AS (
    SELECT *,
        MIN(ABS(delta_filled)) OVER () AS min_abs,
        MAX(ABS(delta_filled)) OVER () AS max_abs
    FROM deltas
)
SELECT
    country_name, year, n_partners, bdis,
    LEAST(bdis, 1.0) AS bdis_norm,
    CASE
        WHEN bdis < 0.2 THEN 'Interdependence'
        WHEN bdis < 0.5 THEN 'Moderate Dependance'
        WHEN bdis < 0.8 THEN 'High Dependance'
        ELSE 'Critical Dependance'
    END AS bdis_level,
    delta_bdis,
    CASE
        WHEN max_abs = 0 THEN 0.5
        ELSE ((ABS(delta_filled) - min_abs) / (max_abs - min_abs) * SIGN(delta_filled) + 1) / 2
    END AS delta_bdis_norm
FROM bounds
ORDER BY country_name, year;

Running query in 'db'

,Success


In [99]:
%%sql
SELECT * FROM analysis.trade_bdi_structural USING SAMPLE 3;

Running query in 'db'

,country_name,year,n_partners,bdis,bdis_norm,bdis_level,delta_bdis,delta_bdis_norm
0,Bolivia (Plurinational State of),2022,189,0.45,0.45,Moderate Dependance,0.01,0.51
1,Mauritania,2017,136,0.76,0.76,High Dependance,0.05,0.55
2,Togo,2015,139,0.66,0.66,High Dependance,NaN,0.50


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 7.2 Visualisation
</div>

In [100]:
%%sql
SELECT country_name, year, bdis, bdis_level, delta_bdis
FROM analysis.trade_bdi_structural
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,bdis,bdis_level,delta_bdis
0,Brazil,2015,0.21,Moderate Dependance,NaN
1,Brazil,2016,0.22,Moderate Dependance,0.01
2,Brazil,2017,0.23,Moderate Dependance,0.01
3,Brazil,2018,0.25,Moderate Dependance,0.02
4,Brazil,2019,0.25,Moderate Dependance,-0.01
...,...,...,...,...,...
78,USA,2021,0.25,Moderate Dependance,-0.00
79,USA,2022,0.24,Moderate Dependance,-0.00
80,USA,2023,0.23,Moderate Dependance,-0.02
81,USA,2024,0.24,Moderate Dependance,0.01


In [101]:
%%sql
SELECT country_name, AVG(bdis) AS avg_bdis
FROM analysis.trade_bdi_structural
GROUP BY country_name
ORDER BY avg_bdis DESC
LIMIT 20;

Running query in 'db'

,country_name,avg_bdis
0,Kuwait,0.92
1,Montserrat,0.89
2,Guinea-Bissau,0.89
3,Djibouti,0.84
4,Sao Tome and Principe,0.84
5,Mali,0.81
6,Suriname,0.80
7,Gambia,0.80
8,Liberia,0.78
9,Burkina Faso,0.78


Heatmap niveau+delta : même information que la table ci-dessus (`bdis`/`delta_bdis` par pays/année), pas de nouvelle donnée -- non répétée.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
## 8. Most Critical Dependencies
</div>

Beyond global concentration (HHI) and bilateral asymmetry (BDI), this section
identifies the most strategically exposed relationships for each country —
capturing not only the dominant partner but the full profile of critical
dependencies that will feed into the bilateral classification scoring.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 8.1 Create Top Dependency TABLE
</div>

`threshold=0.15` : un partenaire est "critique" si `dependency >= 0.15`. Table par (pays, année) : partenaire dominant (`top_partner`, rang 1 par `dependency`), nombre de partenaires critiques, part cumulée de dépendance critique, HHI de dépendance (`SUM(dependency²)`, sur **tous** les partenaires, pas seulement les critiques).

In [102]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_dependency_top AS
WITH bilateral_value AS (
    SELECT country_name, partner_name, year, SUM(value) AS value
    FROM analysis.trade_bilateral
    GROUP BY country_name, partner_name, year
),
enriched AS (
    SELECT b.*, v.value
    FROM analysis.trade_bdi b
    LEFT JOIN bilateral_value v
      ON b.country_name = v.country_name AND b.partner_name = v.partner_name AND b.year = v.year
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY country_name, year ORDER BY dependency DESC) AS partner_rank
    FROM enriched
),
top AS (
    SELECT country_name, year,
           partner_name AS top_partner,
           value AS top_partner_volume,
           bdi AS bdi_top,
           dependency AS dependency_top,
           dependency_reverse
    FROM ranked
    WHERE partner_rank = 1
),
critical AS (
    SELECT country_name, year,
        COUNT(*) FILTER (WHERE dependency >= 0.15) AS n_critical,
        COALESCE(SUM(dependency) FILTER (WHERE dependency >= 0.15), 0) AS critical_share,
        SUM(dependency * dependency) AS dependency_hhi
    FROM enriched
    GROUP BY country_name, year
),
merged AS (
    SELECT t.*, c.n_critical, c.critical_share, c.dependency_hhi
    FROM top t
    JOIN critical c ON t.country_name = c.country_name AND t.year = c.year
),
deltas AS (
    SELECT *,
        bdi_top - LAG(bdi_top) OVER (PARTITION BY country_name ORDER BY year) AS delta_bdi_top,
        dependency_top - LAG(dependency_top) OVER (PARTITION BY country_name ORDER BY year) AS delta_dependency_top,
        n_critical - LAG(n_critical) OVER (PARTITION BY country_name ORDER BY year) AS delta_n_critical,
        critical_share - LAG(critical_share) OVER (PARTITION BY country_name ORDER BY year) AS delta_critical_share
    FROM merged
)
SELECT
    country_name, year, top_partner, top_partner_volume, bdi_top, dependency_top, dependency_reverse,
    n_critical, critical_share, dependency_hhi,
    delta_bdi_top, delta_dependency_top, delta_n_critical, delta_critical_share,
    (dependency_top - MIN(dependency_top) OVER ()) / (MAX(dependency_top) OVER () - MIN(dependency_top) OVER ()) AS dependency_top_norm,
    (n_critical - MIN(n_critical) OVER ()) / (MAX(n_critical) OVER () - MIN(n_critical) OVER ()) AS n_critical_norm,
    (critical_share - MIN(critical_share) OVER ()) / (MAX(critical_share) OVER () - MIN(critical_share) OVER ()) AS critical_share_norm,
    (delta_n_critical - MIN(delta_n_critical) OVER ()) / (MAX(delta_n_critical) OVER () - MIN(delta_n_critical) OVER ()) AS delta_n_critical_norm,
    (delta_dependency_top - MIN(delta_dependency_top) OVER ()) / (MAX(delta_dependency_top) OVER () - MIN(delta_dependency_top) OVER ()) AS delta_dependency_top_norm,
    (delta_critical_share - MIN(delta_critical_share) OVER ()) / (MAX(delta_critical_share) OVER () - MIN(delta_critical_share) OVER ()) AS delta_critical_share_norm,
    CASE WHEN dependency_top < 0.15 THEN 'Low' WHEN dependency_top < 0.25 THEN 'Moderate' WHEN dependency_top < 0.45 THEN 'High' ELSE 'Critical' END AS dependency_top_level,
    CASE WHEN delta_bdi_top IS NULL OR delta_bdi_top = 0 THEN 'Stable' WHEN delta_bdi_top > 0.025 THEN 'Increasing' WHEN delta_bdi_top < -0.025 THEN 'Decreasing' ELSE 'Stable' END AS bdi_top_trend,
    CASE WHEN delta_dependency_top IS NULL OR delta_dependency_top = 0 THEN 'Stable' WHEN delta_dependency_top > 0.025 THEN 'Increasing' WHEN delta_dependency_top < -0.025 THEN 'Decreasing' ELSE 'Stable' END AS dependency_top_trend
FROM deltas
ORDER BY country_name, year;

Running query in 'db'

,Success


In [103]:
%%sql
SELECT * FROM analysis.trade_dependency_top USING SAMPLE 3;

Running query in 'db'

,country_name,year,top_partner,top_partner_volume,bdi_top,dependency_top,dependency_reverse,n_critical,critical_share,dependency_hhi,...,delta_critical_share,dependency_top_norm,n_critical_norm,critical_share_norm,delta_n_critical_norm,delta_dependency_top_norm,delta_critical_share_norm,dependency_top_level,bdi_top_trend,dependency_top_trend
0,Luxembourg,2020,Germany,8.54B,0.24,0.25,0.00,2,0.44,0.12,...,0.00,0.29,0.67,0.50,0.50,0.66,0.66,Moderate,Stable,Stable
1,Nepal,2017,India,6.94B,0.64,0.64,0.01,1,0.64,0.43,...,-0.00,0.75,0.33,0.72,0.50,0.65,0.65,Critical,Stable,Stable
2,State of Palestine,2019,Turkiye,679.8M,0.09,0.09,0.00,0,0.00,0.02,...,0.00,0.10,0.00,0.00,0.50,0.65,0.65,Low,Stable,Stable


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 8.2 Hetmap Top Dependency
</div>

In [104]:
%%sql
SELECT country_name, year, top_partner, dependency_top, bdi_top
FROM analysis.trade_dependency_top
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,top_partner,dependency_top,bdi_top
0,Brazil,2015,China,0.18,0.16
1,Brazil,2016,China,0.18,0.16
2,Brazil,2017,China,0.20,0.18
3,Brazil,2018,China,0.24,0.21
4,Brazil,2019,China,0.24,0.22
...,...,...,...,...,...
78,USA,2021,China,0.15,0.02
79,USA,2022,Canada,0.15,0.48
80,USA,2023,Mexico,0.15,0.46
81,USA,2024,Mexico,0.16,0.45


**Changement de `countries` à noter** : en pandas, la cellule juste après la heatmap (8.2) redéfinit `countries = ['USA', 'China', 'France', 'Egypt', 'Qatar', 'Canada', 'Brazil', 'Tunisia']` (Egypt remplace Saudi Arabia) pour le reste de la section 8. On applique le même changement ici.

In [105]:
%%sql
SET VARIABLE countries = ['USA', 'China', 'France', 'Egypt', 'Qatar', 'Canada', 'Brazil', 'Tunisia'];

Running query in 'db'

,Success


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 8.3 Number of Critical Partners
</div>

In [106]:
%%sql
SELECT
    country_name,
    AVG(n_critical) AS avg_n_critical,
    MAX(n_critical) AS max_n_critical,
    AVG((n_critical > 0)::INT) * 100 AS pct_years_with_critical
FROM analysis.trade_dependency_top
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
GROUP BY country_name
ORDER BY avg_n_critical DESC;

Running query in 'db'

,country_name,avg_n_critical,max_n_critical,pct_years_with_critical
0,Tunisia,1.90,2,100.00
1,Brazil,1.09,2,90.91
2,Canada,1.00,1,100.00
3,USA,0.91,2,72.73
4,Qatar,0.60,1,60.00
5,France,0.40,1,40.00
6,China,0.00,0,0.00
7,Egypt,0.00,0,0.00


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 8.4 Critical Share evolution
</div>

In [107]:
%%sql
SELECT country_name, year, critical_share
FROM analysis.trade_dependency_top
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,critical_share
0,Brazil,2015,0.18
1,Brazil,2016,0.18
2,Brazil,2017,0.20
3,Brazil,2018,0.39
4,Brazil,2019,0.40
...,...,...,...
79,USA,2021,0.00
80,USA,2022,0.00
81,USA,2023,0.31
82,USA,2024,0.16


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">
    
### 8.5 Dependency HHI evolution
</div>

In [108]:
%%sql
SELECT country_name, year, dependency_hhi
FROM analysis.trade_dependency_top
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name, year;

Running query in 'db'

,country_name,year,dependency_hhi
0,Brazil,2015,0.07
1,Brazil,2016,0.07
2,Brazil,2017,0.08
3,Brazil,2018,0.09
4,Brazil,2019,0.10
...,...,...,...
79,USA,2021,0.07
80,USA,2022,0.07
81,USA,2023,0.07
82,USA,2024,0.07


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 9. Bilateral Typology (Annual)
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.1 Merge Tables
</div>

Fusionne `trade_bi_hhi` (par flux) avec `trade_dependency_top` et `trade_bdi_structural` (par pays/année, diffusés sur les deux flux -- comme le `merge how=left` pandas) et `trade_world` (volume, par flux). Contrairement à pandas (merge en 9.1, classification en 9.2, score en 9.3, sur 3 cellules), tout est construit en une seule table ici -- `analysis.trade_bilateral_summary` -- puisque `profile_logic` et `score_flow`/`profile_score` ne dépendent que de colonnes déjà présentes après la fusion.

In [109]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bilateral_summary AS
WITH base AS (
    SELECT h.*,
           d.top_partner, d.top_partner_volume, d.bdi_top, d.dependency_top, d.dependency_reverse,
           d.n_critical, d.critical_share, d.dependency_hhi,
           d.delta_bdi_top, d.delta_dependency_top, d.delta_n_critical, d.delta_critical_share,
           d.dependency_top_norm, d.n_critical_norm, d.critical_share_norm,
           d.delta_n_critical_norm, d.delta_dependency_top_norm, d.delta_critical_share_norm,
           d.dependency_top_level, d.bdi_top_trend, d.dependency_top_trend,
           s.bdis, s.bdis_norm, s.bdis_level, s.delta_bdis,
           w.value
    FROM analysis.trade_bi_hhi h
    LEFT JOIN analysis.trade_dependency_top d ON h.country_name = d.country_name AND h.year = d.year
    LEFT JOIN analysis.trade_bdi_structural s ON h.country_name = s.country_name AND h.year = s.year
    LEFT JOIN analysis.trade_world w ON h.country_name = w.country_name AND h.year = w.year AND h.flow_code = w.flow_code
),
logic AS (
    SELECT *,
        CASE
            WHEN hhi_level = 'Extreme' AND dependency_top_level = 'Critical' THEN 'Critical Dependency'
            WHEN hhi_level IN ('High', 'Extreme') AND dependency_top_level IN ('High', 'Critical')
                 AND hhi_trend = 'Increasing' AND dependency_top_trend = 'Increasing' THEN 'Critical Dependency'
            WHEN hhi_level = 'Extreme' OR dependency_top_level = 'Critical' THEN 'Vulnerable'
            WHEN hhi_level = 'High' AND dependency_top_level = 'High' THEN 'Vulnerable'
            WHEN hhi_level = 'High' AND dependency_top_level = 'Moderate' AND hhi_trend = 'Increasing' THEN 'Vulnerable'
            WHEN dependency_top_level = 'High' AND hhi_level = 'Moderate' AND dependency_top_trend = 'Increasing' THEN 'Vulnerable'
            WHEN hhi_level = 'Moderate' AND dependency_top_level = 'Moderate'
                 AND hhi_trend = 'Increasing' AND dependency_top_trend = 'Increasing' THEN 'Vulnerable'
            WHEN hhi_level IN ('Moderate', 'High') OR dependency_top_level IN ('Moderate', 'High') THEN 'Stable but Concentrated'
            ELSE 'Robust'
        END AS profile_logic,
        0.35 * critical_share_norm + 0.25 * hhi_norm + 0.2 * dependency_top_norm
            + 0.1 * n_critical_norm + 0.1 * bdis_norm AS score_flow
    FROM base
)
SELECT *,
    CASE
        WHEN score_flow < 0.10 THEN 'Robust'
        WHEN score_flow < 0.30 THEN 'Stable but Concentrated'
        WHEN score_flow < 0.50 THEN 'Vulnerable'
        ELSE 'Critical Dependency'
    END AS profile_score
FROM logic;

Running query in 'db'

,Success


In [110]:
%%sql
SELECT * FROM analysis.trade_bilateral_summary USING SAMPLE 3;

Running query in 'db'

,country_name,year,flow_code,n_partners,hhi,delta_hhi,hhi_norm,delta_hhi_norm,hhi_level,hhi_trend,...,bdi_top_trend,dependency_top_trend,bdis,bdis_norm,bdis_level,delta_bdis,value,profile_logic,score_flow,profile_score
0,Bulgaria,2022,M,169,0.05,0.00,0.05,0.50,Low,Stable,...,Stable,Stable,0.27,0.27,Moderate Dependance,0.05,58.09B,Robust,0.07,Robust
1,China,2017,M,218,0.05,-0.00,0.04,0.50,Low,Stable,...,Stable,Stable,0.41,0.41,Moderate Dependance,-0.01,1.84T,Robust,0.09,Robust
2,Portugal,2017,M,184,0.14,-0.01,0.13,0.49,Moderate,Stable,...,Stable,Stable,0.25,0.25,Moderate Dependance,0.02,78.75B,Stable but Concentrated,0.27,Stable but Concentrated


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.2 Logic Profile
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

####  Logic Classification
</div>

`profile_logic` déjà calculée dans la table ci-dessus. Distribution export 2023 puis import 2024 (comme en pandas -- années différentes entre les deux appels).

In [111]:
%%sql
SELECT profile_logic, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_summary
WHERE flow_code = 'X' AND year = 2023
GROUP BY profile_logic
ORDER BY n_countries DESC;

Running query in 'db'

,profile_logic,n_countries
0,Stable but Concentrated,71
1,Vulnerable,43
2,Robust,28
3,Critical Dependency,21


In [112]:
%%sql
SELECT profile_logic, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_summary
WHERE flow_code = 'M' AND year = 2024
GROUP BY profile_logic
ORDER BY n_countries DESC;

Running query in 'db'

,profile_logic,n_countries
0,Stable but Concentrated,76
1,Robust,30
2,Vulnerable,22
3,Critical Dependency,8


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

####  Logic Profile Distribution
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.3 Score & Profile Score
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

####  Score & Profile
</div>

`score_flow`/`profile_score` déjà dans la table (calculés dans la même requête que 9.1/9.2, cf. note plus haut).

In [113]:
%%sql
SELECT profile_score, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_summary
WHERE flow_code = 'X' AND year = 2024
GROUP BY profile_score
ORDER BY n_countries DESC;

Running query in 'db'

,profile_score,n_countries
0,Stable but Concentrated,65
1,Vulnerable,36
2,Robust,26
3,Critical Dependency,9


In [114]:
%%sql
SELECT profile_score, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_summary
WHERE flow_code = 'M' AND year = 2024
GROUP BY profile_score
ORDER BY n_countries DESC;

Running query in 'db'

,profile_score,n_countries
0,Stable but Concentrated,69
1,Vulnerable,34
2,Robust,25
3,Critical Dependency,8


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

####  Profile Distribution
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.4 Final Table
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.4.1 Create Final Table
</div>

Pivote `trade_bilateral_summary` (flow_code M/X → colonnes `_import`/`_export`, `FULL OUTER JOIN` -- un pays peut n'avoir que des lignes M ou X une année donnée, comme le `merge how=outer` pandas), calcule `score_global` (`0.55*import + 0.45*export`) et les deux profils globaux (`profile_score_global` par seuils, `profile_logic_global` combinant les deux logiques). Table : `analysis.trade_bilateral_final`.

In [115]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bilateral_final AS
WITH imp AS (
    SELECT country_name, year, top_partner, dependency_top, dependency_top_norm, dependency_top_level, dependency_top_trend,
           n_critical, n_critical_norm, critical_share, critical_share_norm, bdis, bdis_norm, bdis_level,
           hhi AS hhi_import, hhi_norm AS hhi_import_norm, hhi_level AS hhi_import_level, hhi_trend AS hhi_import_trend,
           delta_hhi AS delta_hhi_import, n_partners AS n_partners_import,
           score_flow AS score_import, profile_score AS profile_score_import, profile_logic AS profile_logic_import,
           value AS value_import
    FROM analysis.trade_bilateral_summary
    WHERE flow_code = 'M'
),
exp AS (
    SELECT country_name, year,
           hhi AS hhi_export, hhi_norm AS hhi_export_norm, hhi_level AS hhi_export_level, hhi_trend AS hhi_export_trend,
           delta_hhi AS delta_hhi_export, n_partners AS n_partners_export,
           score_flow AS score_export, profile_score AS profile_score_export, profile_logic AS profile_logic_export,
           value AS value_export
    FROM analysis.trade_bilateral_summary
    WHERE flow_code = 'X'
),
joined AS (
    SELECT
        COALESCE(i.country_name, e.country_name) AS country_name,
        COALESCE(i.year, e.year) AS year,
        i.top_partner, i.dependency_top, i.dependency_top_norm, i.dependency_top_level, i.dependency_top_trend,
        i.n_critical, i.n_critical_norm, i.critical_share, i.critical_share_norm, i.bdis, i.bdis_norm, i.bdis_level,
        i.hhi_import, i.hhi_import_norm, i.hhi_import_level, i.hhi_import_trend, i.delta_hhi_import, i.n_partners_import,
        i.score_import, i.profile_score_import, i.profile_logic_import, i.value_import,
        e.hhi_export, e.hhi_export_norm, e.hhi_export_level, e.hhi_export_trend, e.delta_hhi_export, e.n_partners_export,
        e.score_export, e.profile_score_export, e.profile_logic_export, e.value_export
    FROM imp i
    FULL OUTER JOIN exp e ON i.country_name = e.country_name AND i.year = e.year
),
scored AS (
    SELECT *,
        value_export + value_import AS total_trade,
        0.55 * score_import + 0.45 * score_export AS score_global
    FROM joined
)
SELECT *,
    CASE
        WHEN score_global < 0.10 THEN 'Robust'
        WHEN score_global < 0.30 THEN 'Stable but Concentrated'
        WHEN score_global < 0.50 THEN 'Vulnerable'
        ELSE 'Critical Dependency'
    END AS profile_score_global,
    CASE
        WHEN profile_logic_import = 'Critical Dependency' AND profile_logic_export = 'Critical Dependency' THEN 'Critical Dependency'
        WHEN profile_logic_import = 'Critical Dependency' OR profile_logic_export = 'Critical Dependency' THEN 'Vulnerable'
        WHEN profile_logic_import = 'Vulnerable' OR profile_logic_export = 'Vulnerable' THEN 'Vulnerable'
        WHEN profile_logic_import = 'Robust' AND profile_logic_export = 'Robust' THEN 'Robust'
        ELSE 'Stable but Concentrated'
    END AS profile_logic_global
FROM scored;

Running query in 'db'

,Success


In [116]:
%%sql
SELECT * FROM analysis.trade_bilateral_final USING SAMPLE 3;

Running query in 'db'

,country_name,year,top_partner,dependency_top,dependency_top_norm,dependency_top_level,dependency_top_trend,n_critical,n_critical_norm,critical_share,...,delta_hhi_export,n_partners_export,score_export,profile_score_export,profile_logic_export,value_export,total_trade,score_global,profile_score_global,profile_logic_global
0,Andorra,2022,Spain,0.68,0.80,Critical,Increasing,1,0.33,0.68,...,0.26,116,0.63,Critical Dependency,Critical Dependency,386.1M,2.25B,0.61,Critical Dependency,Critical Dependency
1,Mongolia,2015,China,0.62,0.73,Critical,Stable,1,0.33,0.62,...,NaN,69,0.65,Critical Dependency,Critical Dependency,4.67B,8.47B,0.59,Critical Dependency,Vulnerable
2,Mongolia,2016,China,0.60,0.70,Critical,Stable,1,0.33,0.60,...,-0.05,74,0.63,Critical Dependency,Critical Dependency,4.92B,8.26B,0.57,Critical Dependency,Vulnerable


In [117]:
%%sql
SELECT profile_logic_global, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_final
WHERE year = 2024
GROUP BY profile_logic_global
ORDER BY n_countries DESC;

Running query in 'db'

,profile_logic_global,n_countries
0,Stable but Concentrated,66
1,Vulnerable,41
2,Robust,22
3,Critical Dependency,7


In [118]:
%%sql
SELECT profile_score_global, COUNT(*) AS n_countries
FROM analysis.trade_bilateral_final
WHERE year = 2024
GROUP BY profile_score_global
ORDER BY n_countries DESC;

Running query in 'db'

,profile_score_global,n_countries
0,Stable but Concentrated,70
1,Vulnerable,33
2,Robust,25
3,Critical Dependency,8


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.4.2 Global Score
</div>

In [119]:
%%sql
SELECT country_name, score_global, profile_score_global
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY score_global DESC;

Running query in 'db'

,country_name,score_global,profile_score_global
0,Canada,0.56,Critical Dependency
1,Brazil,0.25,Stable but Concentrated
2,Qatar,0.21,Stable but Concentrated
3,Tunisia,0.19,Stable but Concentrated
4,USA,0.17,Stable but Concentrated
5,Egypt,0.09,Robust
6,China,0.07,Robust
7,France,0.06,Robust


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.4.3 Import vs Export Vulnerability
</div>

Remplacé par une classification en 4 quadrants (seuil 0.50 sur les deux scores, comme les lignes de seuil du scatter pandas) : comptage mondial + détail pour `countries`.

In [120]:
%%sql
SELECT
    CASE
        WHEN score_import >= 0.50 AND score_export >= 0.50 THEN 'Dual Exposed'
        WHEN score_import >= 0.50 AND score_export <  0.50 THEN 'Import Exposed'
        WHEN score_import <  0.50 AND score_export >= 0.50 THEN 'Export Exposed'
        ELSE 'Balanced Low Risk'
    END AS quadrant,
    COUNT(*) AS n_countries
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND score_import IS NOT NULL AND score_export IS NOT NULL
  AND NOT isnan(score_import) AND NOT isnan(score_export)
GROUP BY quadrant
ORDER BY n_countries DESC;

Running query in 'db'

,quadrant,n_countries
0,Balanced Low Risk,127
1,Dual Exposed,7
2,Export Exposed,1


In [121]:
%%sql
SELECT country_name, score_import, score_export, profile_score_global
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY country_name;

Running query in 'db'

,country_name,score_import,score_export,profile_score_global
0,Brazil,0.25,0.25,Stable but Concentrated
1,Canada,0.52,0.60,Critical Dependency
2,China,0.07,0.08,Robust
3,Egypt,0.09,0.08,Robust
4,France,0.06,0.06,Robust
5,Qatar,0.21,0.22,Stable but Concentrated
6,Tunisia,0.19,0.20,Stable but Concentrated
7,USA,0.17,0.17,Stable but Concentrated


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.4.4 Risk matrix (HHI vs Dependency)
</div>

Remplacé par une classification en 4 quadrants selon les médianes (HHI moyen import/export vs `critical_share`, année 2024).

In [122]:
%%sql
WITH y AS (
    SELECT *, (hhi_import + hhi_export) / 2 AS hhi_avg
    FROM analysis.trade_bilateral_final
    WHERE year = 2024 AND score_global IS NOT NULL
),
medians AS (
    SELECT MEDIAN(hhi_avg) AS median_hhi, MEDIAN(critical_share) AS median_crit FROM y
)
SELECT
    CASE
        WHEN hhi_avg >= median_hhi AND critical_share >= median_crit THEN 'Dual Risk'
        WHEN hhi_avg >= median_hhi AND critical_share <  median_crit THEN 'HHI Driven'
        WHEN hhi_avg <  median_hhi AND critical_share >= median_crit THEN 'Critical Share Driven'
        ELSE 'Low Exposure'
    END AS quadrant,
    COUNT(*) AS n_countries
FROM y CROSS JOIN medians
GROUP BY quadrant
ORDER BY n_countries DESC;

Running query in 'db'

,quadrant,n_countries
0,Low Exposure,53
1,Dual Risk,53
2,Critical Share Driven,15
3,HHI Driven,15


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.5 Results And Stats
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.5.1 Results
</div>

In [123]:
%%sql
SELECT country_name, score_global, score_import, score_export, profile_score_global, profile_logic_global
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY score_global;

Running query in 'db'

,country_name,score_global,score_import,score_export,profile_score_global,profile_logic_global
0,France,0.06,0.06,0.06,Robust,Robust
1,China,0.07,0.07,0.08,Robust,Robust
2,Egypt,0.09,0.09,0.08,Robust,Robust
3,USA,0.17,0.17,0.17,Stable but Concentrated,Stable but Concentrated
4,Tunisia,0.19,0.19,0.20,Stable but Concentrated,Stable but Concentrated
5,Qatar,0.21,0.21,0.22,Stable but Concentrated,Stable but Concentrated
6,Brazil,0.25,0.25,0.25,Stable but Concentrated,Stable but Concentrated
7,Canada,0.56,0.52,0.60,Critical Dependency,Critical Dependency


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.5.2 Profiles Distribution
</div>

In [124]:
%%sql
SELECT year,
    COUNT(*) FILTER (WHERE profile_logic_global = 'Robust')::DOUBLE / COUNT(*) * 100 AS pct_robust,
    COUNT(*) FILTER (WHERE profile_logic_global = 'Stable but Concentrated')::DOUBLE / COUNT(*) * 100 AS pct_stable,
    COUNT(*) FILTER (WHERE profile_logic_global = 'Vulnerable')::DOUBLE / COUNT(*) * 100 AS pct_vulnerable,
    COUNT(*) FILTER (WHERE profile_logic_global = 'Critical Dependency')::DOUBLE / COUNT(*) * 100 AS pct_critical
FROM analysis.trade_bilateral_final
GROUP BY year
ORDER BY year;

Running query in 'db'

,year,pct_robust,pct_stable,pct_vulnerable,pct_critical
0,2015,15.43,45.71,32.00,6.86
1,2016,13.56,45.76,30.51,10.17
2,2017,15.00,41.67,36.67,6.67
3,2018,12.99,46.89,32.20,7.91
4,2019,13.45,45.61,33.92,7.02
5,2020,14.29,44.64,35.12,5.95
6,2021,12.50,45.83,34.52,7.14
7,2022,15.15,44.85,32.73,7.27
8,2023,14.11,41.72,36.81,7.36
9,2024,16.18,48.53,30.15,5.15


Distribution des scores (remplace histogramme+KDE) : stats descriptives par métrique, année 2024.

**Cas limite (`NaN`)** : quand un pays n'a qu'**un seul** partenaire sur un flux (`n_partners=1`), `hhi_norm = (hhi - 1/n_partners) / (1 - 1/n_partners)` devient `0/0 = NaN` -- déjà présent dans `df_bi_hhi.csv` pandas (Montserrat, export, 2024 -- vérifié), donc pas une divergence introduite ici. Contrairement à pandas où `.mean()`/`.std()` ignorent silencieusement les `NaN`, DuckDB `STDDEV` lève une erreur s'il en rencontre -- `NOT isnan(...)` est ajouté dans les requêtes d'agrégat/classement/quadrants qui suivent pour rester correct (le tri DuckDB place `NaN` en tête d'un `ORDER BY DESC`, ce qui aurait classé ce pays #1 "most vulnerable" à tort).

In [125]:
%%sql
SELECT 'score_import' AS metric, AVG(score_import) AS mean, MEDIAN(score_import) AS median, STDDEV(score_import) AS stddev
FROM analysis.trade_bilateral_final WHERE year = 2024 AND NOT isnan(score_import)
UNION ALL
SELECT 'score_export', AVG(score_export), MEDIAN(score_export), STDDEV(score_export)
FROM analysis.trade_bilateral_final WHERE year = 2024 AND NOT isnan(score_export)
UNION ALL
SELECT 'score_global', AVG(score_global), MEDIAN(score_global), STDDEV(score_global)
FROM analysis.trade_bilateral_final WHERE year = 2024 AND NOT isnan(score_global);

Running query in 'db'

,metric,mean,median,stddev
0,score_import,0.25,0.23,0.15
1,score_export,0.25,0.24,0.15
2,score_global,0.25,0.23,0.14


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.5.3 Top Robust/Vulnerable countries
</div>

In [126]:
%%sql
SELECT country_name, score_global, profile_score_global
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND score_global IS NOT NULL AND NOT isnan(score_global)
ORDER BY score_global DESC
LIMIT 15;

Running query in 'db'

,country_name,score_global,profile_score_global
0,Cayman Isds,0.75,Critical Dependency
1,Bahamas,0.73,Critical Dependency
2,Bermuda,0.71,Critical Dependency
3,Lesotho,0.65,Critical Dependency
4,Mexico,0.56,Critical Dependency
5,Canada,0.56,Critical Dependency
6,Paraguay,0.51,Critical Dependency
7,Cabo Verde,0.47,Vulnerable
8,Antigua and Barbuda,0.47,Vulnerable
9,Barbados,0.46,Vulnerable


In [127]:
%%sql
SELECT country_name, score_global, profile_score_global
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND score_global IS NOT NULL AND NOT isnan(score_global)
ORDER BY score_global ASC
LIMIT 15;

Running query in 'db'

,country_name,score_global,profile_score_global
0,Germany,0.05,Robust
1,France,0.06,Robust
2,Finland,0.06,Robust
3,Sweden,0.07,Robust
4,Spain,0.07,Robust
5,United Kingdom,0.07,Robust
6,Italy,0.07,Robust
7,Lithuania,0.07,Robust
8,Turkiye,0.07,Robust
9,Serbia,0.07,Robust


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.6 PROFILES CONVERGENCE/DIVERGENCE
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.6.1 Convergence
</div>

Ajoute `convergence` (score profile == logic profile) à `analysis.trade_bilateral_final` -- extension en place, comme en 4.5/4.6 (`CREATE OR REPLACE TABLE t AS SELECT * FROM t`).

In [128]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bilateral_final AS
SELECT *, (profile_score_global = profile_logic_global) AS convergence
FROM analysis.trade_bilateral_final;

Running query in 'db'

,Success


In [129]:
%%sql
SELECT profile_score_global, profile_logic_global, COUNT(*) AS n
FROM analysis.trade_bilateral_final
WHERE year = 2024
GROUP BY profile_score_global, profile_logic_global
ORDER BY profile_score_global, profile_logic_global;

Running query in 'db'

,profile_score_global,profile_logic_global,n
0,Critical Dependency,Critical Dependency,6
1,Critical Dependency,Vulnerable,2
2,Robust,Robust,21
3,Robust,Stable but Concentrated,4
4,Stable but Concentrated,Robust,1
5,Stable but Concentrated,Stable but Concentrated,50
6,Stable but Concentrated,Vulnerable,19
7,Vulnerable,Critical Dependency,1
8,Vulnerable,Stable but Concentrated,12
9,Vulnerable,Vulnerable,20


In [130]:
%%sql
SELECT AVG(convergence::INT) * 100 AS convergence_rate_pct
FROM analysis.trade_bilateral_final
WHERE year = 2024;

Running query in 'db'

,convergence_rate_pct
0,71.32


In [131]:
%%sql
SELECT profile_score_global, profile_logic_global, COUNT(*) AS n
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND NOT convergence
GROUP BY profile_score_global, profile_logic_global
ORDER BY n DESC
LIMIT 10;

Running query in 'db'

,profile_score_global,profile_logic_global,n
0,Stable but Concentrated,Vulnerable,19
1,Vulnerable,Stable but Concentrated,12
2,Robust,Stable but Concentrated,4
3,Critical Dependency,Vulnerable,2
4,Vulnerable,Critical Dependency,1
5,Stable but Concentrated,Robust,1


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.6.2 Divergence Type, Level & Automatic Interpretation
</div>

`divergence_level` = distance entre les rangs des deux profils (0=Robust ... 3=Critical Dependency). `diagnostic_note` réplique `interpret_divergence` -- en pandas les tests `'X' in str(profile)` équivalent à une égalité stricte ici (aucun nom de profil n'est sous-chaîne d'un autre).

In [132]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_bilateral_final AS
WITH ranks AS (
    SELECT *,
        CASE profile_score_global WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END AS score_rank,
        CASE profile_logic_global WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END AS logic_rank
    FROM analysis.trade_bilateral_final
)
SELECT * EXCLUDE (score_rank, logic_rank),
    ABS(score_rank - logic_rank) AS divergence_level,
    CASE ABS(score_rank - logic_rank)
        WHEN 0 THEN 'Convergence'
        WHEN 1 THEN 'Minor Divergence'
        WHEN 2 THEN 'Moderate Divergence'
        ELSE 'Major Divergence'
    END AS convergence_type,
    CASE
        WHEN convergence THEN 'Consistent diagnosis'
        WHEN profile_logic_global = 'Robust' AND profile_score_global = 'Vulnerable' THEN 'Emerging risk -- deterioration captured by score'
        WHEN profile_logic_global = 'Vulnerable' AND profile_score_global = 'Stable but Concentrated' THEN 'Structural concentration but no worsening trend'
        WHEN profile_score_global = 'Critical Dependency' THEN 'Rapid dependency build-up'
        ELSE 'Structural transition phase'
    END AS diagnostic_note
FROM ranks;

Running query in 'db'

,Success


In [133]:
%%sql
SELECT country_name, profile_score_global, profile_logic_global, divergence_level, convergence_type, diagnostic_note
FROM analysis.trade_bilateral_final
WHERE year = 2024 AND divergence_level >= 2
ORDER BY divergence_level DESC, country_name;

Running query in 'db'

,country_name,profile_score_global,profile_logic_global,divergence_level,convergence_type,diagnostic_note


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 9.6.3 Profiles Evolution
</div>

Timeline pour le pays choisi (`country`, section 3.1) : profils + indicateurs clés année par année -- remplace la heatmap + double axe pandas (et l'animation Plotly, purement interactive, aucune nouvelle donnée).

In [134]:
%%sql
SELECT year, profile_score_global, profile_logic_global, profile_score_import, profile_score_export,
       hhi_import, hhi_export, critical_share, convergence
FROM analysis.trade_bilateral_final
WHERE country_name = getvariable('country')
ORDER BY year;

Running query in 'db'

,year,profile_score_global,profile_logic_global,profile_score_import,profile_score_export,hhi_import,hhi_export,critical_share,convergence
0,2015,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.09,0.08,0.32,True
1,2016,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.09,0.08,0.16,True
2,2017,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.09,0.08,0.17,True
3,2018,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.09,0.07,0.16,True
4,2019,Robust,Stable but Concentrated,Robust,Robust,0.08,0.07,0.00,False
5,2020,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.08,0.07,0.15,True
6,2021,Robust,Robust,Robust,Robust,0.08,0.07,0.00,True
7,2022,Robust,Robust,Robust,Robust,0.08,0.07,0.00,True
8,2023,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.07,0.07,0.31,True
9,2024,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,Stable but Concentrated,0.07,0.07,0.16,True


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 9.7 EXPORT DATAFRAME
</div>

Rien à coder -- `analysis.trade_bilateral_final` est déjà persistée dans `data/sql/trade_analysis_sql.duckdb`, à jour avec toutes les colonnes de 9.1 à 9.6.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 10. Bilateral Typology (Full Period)
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.1 Full Period Table
</div>

**Full Period Aggregation : Median/Mean**

The median is preferred over the mean for full-period aggregation because
the 2020 COVID shock introduced extreme outliers in all bilateral indicators —
HHI spikes, dependency surges, critical share collapses.
The median, by construction, ignores these extremes and captures the
**typical structural behavior** of each country across the period —
making it the most analytically defensible choice for a full-period profile.
Exception : n_critical uses mean because it is a discrete count
where the median collapses to 0 or 1, losing discriminatory power.

`analysis.trade_full_period` : agrégation médiane (moyenne pour `n_critical`) de `trade_bilateral_final` par pays, sur toutes les années. `MEDIAN(...) FILTER (WHERE NOT isnan(...))` reproduit le `skipna=True` par défaut de pandas -- important ici vu le cas `NaN` documenté en 9.5.2 (pays à 1 seul partenaire une année donnée). `top_partner` = `MODE()` DuckDB (le partenaire le plus fréquent sur la période) -- **écart confirmé** : sur 188 pays, 2 (Nigeria, Qatar) ont un partenaire dominant ex æquo sur la période et le départage diffère de `.mode().iloc[0]` pandas (ordre alphabétique) ; le reste (186 pays) est identique.

In [135]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_full_period AS
WITH agg AS (
    SELECT
        country_name,
        MEDIAN(hhi_import) AS hhi_import,
        MEDIAN(hhi_export) AS hhi_export,
        MEDIAN(hhi_import_norm) FILTER (WHERE NOT isnan(hhi_import_norm)) AS hhi_import_norm,
        MEDIAN(hhi_export_norm) FILTER (WHERE NOT isnan(hhi_export_norm)) AS hhi_export_norm,
        MEDIAN(critical_share) AS critical_share,
        MEDIAN(critical_share_norm) AS critical_share_norm,
        MEDIAN(dependency_top) AS dependency_top,
        MEDIAN(dependency_top_norm) AS dependency_top_norm,
        AVG(n_critical) AS n_critical,
        AVG(n_critical_norm) AS n_critical_norm,
        MEDIAN(bdis) AS bdis,
        MEDIAN(bdis_norm) AS bdis_norm,
        MEDIAN(score_import) FILTER (WHERE NOT isnan(score_import)) AS score_import,
        MEDIAN(score_export) FILTER (WHERE NOT isnan(score_export)) AS score_export,
        MEDIAN(score_global) FILTER (WHERE NOT isnan(score_global)) AS score_global
    FROM analysis.trade_bilateral_final
    GROUP BY country_name
),
top_partner_mode AS (
    SELECT country_name, MODE(top_partner) AS top_partner
    FROM analysis.trade_bilateral_final
    GROUP BY country_name
)
SELECT a.*, t.top_partner,
    (a.hhi_import + a.hhi_export) / 2 AS hhi_avg,
    (a.hhi_import_norm + a.hhi_export_norm) / 2 AS hhi_norm_avg
FROM agg a
LEFT JOIN top_partner_mode t ON a.country_name = t.country_name;

Running query in 'db'

,Success


In [136]:
%%sql
SELECT * FROM analysis.trade_full_period USING SAMPLE 3;

Running query in 'db'

,country_name,hhi_import,hhi_export,hhi_import_norm,hhi_export_norm,critical_share,critical_share_norm,dependency_top,dependency_top_norm,n_critical,n_critical_norm,bdis,bdis_norm,score_import,score_export,score_global,top_partner,hhi_avg,hhi_norm_avg
0,Niger,0.10,0.19,0.09,0.17,0.36,0.40,0.21,0.24,1.60,0.53,0.63,0.63,0.34,0.35,0.34,France,0.14,0.13
1,Panama,0.11,0.08,0.10,0.07,0.19,0.22,0.19,0.22,1.36,0.45,0.64,0.64,0.24,0.24,0.24,USA,0.10,0.09
2,State of Palestine,0.32,0.71,0.32,0.70,0.00,0.00,0.09,0.10,0.00,0.00,0.35,0.35,0.13,0.23,0.18,Turkiye,0.52,0.51


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.2 Score & Classification
</div>

In [137]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_full_period AS
WITH scored AS (
    SELECT *,
        0.35 * critical_share_norm + 0.25 * hhi_norm_avg + 0.20 * dependency_top_norm
            + 0.10 * n_critical_norm + 0.10 * bdis_norm AS score_full_periode
    FROM analysis.trade_full_period
)
SELECT *,
    CASE
        WHEN score_full_periode < 0.10 THEN 'Robust'
        WHEN score_full_periode < 0.30 THEN 'Stable but Concentrated'
        WHEN score_full_periode < 0.50 THEN 'Vulnerable'
        ELSE 'Critical Dependency'
    END AS profile_full_periode
FROM scored;

Running query in 'db'

,Success


In [138]:
%%sql
SELECT profile_full_periode, COUNT(*) AS n_countries
FROM analysis.trade_full_period
GROUP BY profile_full_periode
ORDER BY n_countries DESC;

Running query in 'db'

,profile_full_periode,n_countries
0,Stable but Concentrated,84
1,Vulnerable,60
2,Robust,26
3,Critical Dependency,18


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.3 Profile Stability
</div>

`stability_score` = fréquence du profil annuel le plus courant sur la période (le pays a-t-il gardé le même profil chaque année, ou changé souvent ?). `convergence_rate` = moyenne de `convergence` (9.6.1) sur toutes les années du pays. **Même type d'écart qu'en 10.1** : `dominant_profile` diverge pour 2 pays (Georgia, Ireland) à égalité parfaite (`stability_score = 0.5`, deux profils sur autant d'années chacun) -- départage par ordre alphabétique ici (`ORDER BY ... profile_score_global ASC`) au lieu de l'ordre d'apparition pandas (`value_counts().index[0]`).

In [139]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_full_period AS
WITH counts AS (
    SELECT country_name, profile_score_global, COUNT(*) AS cnt
    FROM analysis.trade_bilateral_final
    GROUP BY country_name, profile_score_global
),
totals AS (
    SELECT country_name, COUNT(*) AS total_years
    FROM analysis.trade_bilateral_final
    GROUP BY country_name
),
ranked AS (
    SELECT c.country_name, c.profile_score_global, c.cnt, t.total_years,
           ROW_NUMBER() OVER (PARTITION BY c.country_name ORDER BY c.cnt DESC, c.profile_score_global ASC) AS rnk
    FROM counts c
    JOIN totals t ON c.country_name = t.country_name
),
stability AS (
    SELECT country_name, profile_score_global AS dominant_profile, cnt::DOUBLE / total_years AS stability_score
    FROM ranked
    WHERE rnk = 1
),
convergence AS (
    SELECT country_name, AVG(convergence::INT) AS convergence_rate
    FROM analysis.trade_bilateral_final
    GROUP BY country_name
)
SELECT f.*, s.stability_score, c.convergence_rate, s.dominant_profile,
    CASE
        WHEN s.stability_score >= 0.80 THEN 'Very Stable'
        WHEN s.stability_score >= 0.60 THEN 'Stable'
        WHEN s.stability_score >= 0.40 THEN 'Moderate'
        ELSE 'Unstable'
    END AS stability_label
FROM analysis.trade_full_period f
LEFT JOIN stability s ON f.country_name = s.country_name
LEFT JOIN convergence c ON f.country_name = c.country_name;

Running query in 'db'

,Success


In [140]:
%%sql
SELECT * FROM analysis.trade_full_period USING SAMPLE 3;

Running query in 'db'

,country_name,hhi_import,hhi_export,hhi_import_norm,hhi_export_norm,critical_share,critical_share_norm,dependency_top,dependency_top_norm,n_critical,...,score_global,top_partner,hhi_avg,hhi_norm_avg,score_full_periode,profile_full_periode,stability_score,convergence_rate,dominant_profile,stability_label
0,Mexico,0.24,0.64,0.24,0.64,0.62,0.69,0.62,0.72,1.00,...,0.56,USA,0.44,0.44,0.57,Critical Dependency,1.00,0.36,Critical Dependency,Very Stable
1,Haiti,0.13,0.34,0.13,0.33,0.32,0.36,0.32,0.38,1.00,...,0.34,USA,0.24,0.23,0.34,Vulnerable,1.00,1.00,Vulnerable,Very Stable
2,Sri Lanka,0.10,0.09,0.10,0.08,0.15,0.17,0.15,0.18,0.78,...,0.21,India,0.09,0.09,0.20,Stable but Concentrated,0.89,0.89,Stable but Concentrated,Very Stable


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.4 Profile Drift — Full Period vs Recent
</div>

`score_recent` = médiane de `score_global` sur les 3 dernières années. `drift` compare le rang du profil récent à celui du profil plein-période (0=Robust ... 3=Critical Dependency) : positif = dégradation, négatif = amélioration.

In [141]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_full_period AS
WITH recent_years AS (
    SELECT DISTINCT year FROM analysis.trade_bilateral_final ORDER BY year DESC LIMIT 3
),
recent AS (
    SELECT country_name, MEDIAN(score_global) FILTER (WHERE NOT isnan(score_global)) AS score_recent
    FROM analysis.trade_bilateral_final
    WHERE year IN (SELECT year FROM recent_years)
    GROUP BY country_name
),
recent_classified AS (
    SELECT *,
        CASE
            WHEN score_recent < 0.10 THEN 'Robust'
            WHEN score_recent < 0.30 THEN 'Stable but Concentrated'
            WHEN score_recent < 0.50 THEN 'Vulnerable'
            ELSE 'Critical Dependency'
        END AS profile_recent
    FROM recent
),
ranks AS (
    SELECT f.*, r.score_recent, r.profile_recent,
        CASE r.profile_recent WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END AS recent_rank,
        CASE f.profile_full_periode WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END AS full_rank
    FROM analysis.trade_full_period f
    LEFT JOIN recent_classified r ON f.country_name = r.country_name
)
SELECT * EXCLUDE (recent_rank, full_rank),
    (recent_rank - full_rank) AS drift_numeric,
    CASE
        WHEN (recent_rank - full_rank) > 0 THEN 'Worsening'
        WHEN (recent_rank - full_rank) < 0 THEN 'Improving'
        ELSE 'Stable'
    END AS drift,
    ABS(recent_rank - full_rank) AS drift_magnitude
FROM ranks;

Running query in 'db'

,Success


In [142]:
%%sql
SELECT drift, COUNT(*) AS n_countries
FROM analysis.trade_full_period
GROUP BY drift
ORDER BY n_countries DESC;

Running query in 'db'

,drift,n_countries
0,Stable,167
1,Improving,13
2,Worsening,8


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.5 Annual Divergence Analysis
</div>

Pour chaque (pays, année), compare `profile_score_global` (annuel) au `profile_full_periode` (plein-période) -- fréquence de divergence par pays, classée en 4 catégories.

In [143]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_annual_divergence AS
WITH full_map AS (
    SELECT country_name, profile_full_periode,
        CASE profile_full_periode WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END AS full_rank
    FROM analysis.trade_full_period
),
div AS (
    SELECT b.country_name, b.year, b.profile_score_global, f.profile_full_periode,
        (b.profile_score_global != f.profile_full_periode) AS diverge_from_full_periode,
        ABS(
            CASE b.profile_score_global WHEN 'Robust' THEN 0 WHEN 'Stable but Concentrated' THEN 1 WHEN 'Vulnerable' THEN 2 WHEN 'Critical Dependency' THEN 3 END
            - f.full_rank
        ) AS diverge_magnitude
    FROM analysis.trade_bilateral_final b
    LEFT JOIN full_map f ON b.country_name = f.country_name
)
SELECT
    country_name,
    AVG(diverge_from_full_periode::INT) AS divergence_freq,
    SUM(diverge_from_full_periode::INT) AS divergence_count,
    AVG(diverge_magnitude) AS avg_magnitude,
    LIST(year ORDER BY year) FILTER (WHERE diverge_from_full_periode) AS divergence_years,
    CASE
        WHEN AVG(diverge_from_full_periode::INT) = 0.0 THEN 'Never'
        WHEN AVG(diverge_from_full_periode::INT) <= 0.20 THEN 'Rare'
        WHEN AVG(diverge_from_full_periode::INT) <= 0.50 THEN 'Frequent'
        ELSE 'Systematic'
    END AS divergence_category
FROM div
GROUP BY country_name;

Running query in 'db'

,Success


In [144]:
%%sql
SELECT divergence_category, COUNT(*) AS n_countries
FROM analysis.trade_annual_divergence
GROUP BY divergence_category
ORDER BY n_countries DESC;

Running query in 'db'

,divergence_category,n_countries
0,Never,91
1,Frequent,47
2,Rare,46
3,Systematic,4


In [145]:
%%sql
SELECT country_name, divergence_freq, divergence_years
FROM analysis.trade_annual_divergence
WHERE divergence_category = 'Systematic'
ORDER BY divergence_freq DESC;

Running query in 'db'

,country_name,divergence_freq,divergence_years
0,Nigeria,0.70,"[2015, 2016, 2017, 2018, 2022, 2023, 2024]"
1,Norway,0.64,"[2016, 2017, 2018, 2019, 2020, 2022, 2024]"
2,Japan,0.64,"[2015, 2016, 2017, 2018, 2019, 2023, 2024]"
3,Paraguay,0.55,"[2015, 2016, 2020, 2023, 2024, 2025]"


Divergence par année (chocs systémiques -- même logique que ci-dessus, agrégée par année plutôt que par pays).

In [146]:
%%sql
SELECT b.year,
    AVG((b.profile_score_global != f.profile_full_periode)::INT) AS divergence_rate
FROM analysis.trade_bilateral_final b
LEFT JOIN analysis.trade_full_period f ON b.country_name = f.country_name
GROUP BY b.year
ORDER BY b.year;

Running query in 'db'

,year,divergence_rate
0,2015,0.18
1,2016,0.12
2,2017,0.11
3,2018,0.15
4,2019,0.11
5,2020,0.10
6,2021,0.11
7,2022,0.13
8,2023,0.13
9,2024,0.17


**Note sur le commentaire pandas** : le notebook pandas affirme (cellule d'insights 10.6.5) "2015 shows the highest annual divergence rate (20.7%)" et un minimum en 2021 (8.3%). En recalculant directement depuis les CSV pandas sauvegardés (`df_final.csv` + `df_full_periode.csv`), on obtient 2015 ≈ 18.3% et un minimum en 2020 ≈ 10.1% -- identique au résultat SQL ci-dessus au pourcentage près. Le texte d'insight pandas est donc probablement resté d'un run antérieur des données, pas une divergence introduite ici.

In [147]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_full_period AS
SELECT f.*, d.divergence_freq, d.divergence_count, d.avg_magnitude, d.divergence_category
FROM analysis.trade_full_period f
LEFT JOIN analysis.trade_annual_divergence d ON f.country_name = d.country_name;

Running query in 'db'

,Success


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.6 Visualisation
</div>

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 10.6.1 Ranking
</div>

In [148]:
%%sql
SELECT country_name, score_full_periode, profile_full_periode
FROM analysis.trade_full_period
WHERE score_full_periode IS NOT NULL AND NOT isnan(score_full_periode)
ORDER BY score_full_periode DESC
LIMIT 15;

Running query in 'db'

,country_name,score_full_periode,profile_full_periode
0,Bahamas,0.73,Critical Dependency
1,Cayman Isds,0.72,Critical Dependency
2,Bhutan,0.71,Critical Dependency
3,Bermuda,0.65,Critical Dependency
4,Greenland,0.63,Critical Dependency
5,Eswatini,0.63,Critical Dependency
6,Mongolia,0.61,Critical Dependency
7,Saint Kitts and Nevis,0.59,Critical Dependency
8,Lesotho,0.59,Critical Dependency
9,Nepal,0.57,Critical Dependency


In [149]:
%%sql
SELECT country_name, score_full_periode, profile_full_periode
FROM analysis.trade_full_period
WHERE score_full_periode IS NOT NULL AND NOT isnan(score_full_periode)
ORDER BY score_full_periode ASC
LIMIT 15;

Running query in 'db'

,country_name,score_full_periode,profile_full_periode
0,Germany,0.05,Robust
1,Bulgaria,0.06,Robust
2,Spain,0.07,Robust
3,United Kingdom,0.07,Robust
4,Finland,0.07,Robust
5,Serbia,0.07,Robust
6,Turkiye,0.07,Robust
7,Sweden,0.07,Robust
8,Lithuania,0.07,Robust
9,Italy,0.07,Robust


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 10.6.2 Stability
</div>

In [150]:
%%sql
SELECT country_name, profile_full_periode, score_full_periode, stability_label,
       convergence_rate, profile_recent, drift
FROM analysis.trade_full_period
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY score_full_periode DESC;

Running query in 'db'

,country_name,profile_full_periode,score_full_periode,stability_label,convergence_rate,profile_recent,drift
0,Canada,Critical Dependency,0.56,Very Stable,0.91,Critical Dependency,Stable
1,Tunisia,Vulnerable,0.31,Stable,0.40,Stable but Concentrated,Improving
2,Brazil,Stable but Concentrated,0.24,Stable,0.64,Stable but Concentrated,Stable
3,Qatar,Stable but Concentrated,0.18,Very Stable,1.00,Stable but Concentrated,Stable
4,USA,Stable but Concentrated,0.17,Stable,0.91,Stable but Concentrated,Stable
5,China,Robust,0.08,Very Stable,1.00,Robust,Stable
6,Egypt,Robust,0.08,Very Stable,1.00,Robust,Stable
7,France,Robust,0.08,Stable,1.00,Robust,Stable


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 10.6.3 Divergence Timeline
</div>

Déjà montrée en 10.5 (divergence par année) -- non répétée.

<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 10.6.4 Drift
</div>

In [151]:
%%sql
SELECT country_name, score_full_periode, score_recent, drift, drift_magnitude
FROM analysis.trade_full_period
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY drift_magnitude DESC;

Running query in 'db'

,country_name,score_full_periode,score_recent,drift,drift_magnitude
0,Tunisia,0.31,0.24,Improving,1
1,USA,0.17,0.17,Stable,0
2,Brazil,0.24,0.25,Stable,0
3,Canada,0.56,0.56,Stable,0
4,China,0.08,0.07,Stable,0
5,Egypt,0.08,0.09,Stable,0
6,France,0.08,0.06,Stable,0
7,Qatar,0.18,0.21,Stable,0


<div style="background-color: #fef9e7; padding-left: 20px; border: 1px solid; border-radius: 5px;">

#### 10.6.5 Insights
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 10.7 Export Table
</div>

Rien à coder -- `analysis.trade_full_period` est déjà persistée dans `data/sql/trade_analysis_sql.duckdb`, à jour avec toutes les colonnes de 10.1 à 10.5.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 11. Bilateral Trade Insights
</div>

Synthèse courte, basée sur les tables construites en G.5-G.10 (pas une reprise du texte pandas -- certains chiffres y sont obsolètes, cf. note en 10.5) :

- **Concentration structurelle** : la plupart des pays dépendent d'un petit nombre de partenaires (`trade_bi_hhi`) -- rarement diversifiés au sens du seuil `hhi < 0.08`.
- **Asymétrie systématique** : le BDI (`trade_bdi`) est presque toujours loin de 0 -- un partenaire domine quasiment toujours la relation bilatérale, dans les deux sens (import et export).
- **Profils globalement stables** : `trade_full_period.stability_label` montre que la majorité des pays gardent le même profil annuel sur toute la période -- les changements de profil sont rares et abrupts plutôt que graduels.
- **Convergence méthodologique élevée** : `profile_score_global` et `profile_logic_global` (`trade_bilateral_final`) s'accordent pour la grande majorité des pays -- les cas divergents (`convergence_type`) signalent des pays en transition structurelle, pas des erreurs de classification.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 12. Bilateral Trade Conclusions
</div>

Cette section G reconstruit en SQL toute la chaîne d'analyse bilatérale de pandas : structure des partenaires et concentration (G.5), dépendance asymétrique (G.6-G.8), typologie annuelle puis plein-période (G.9-G.10) -- huit tables persistées (`trade_bilateral`, `trade_bi_hhi`, `trade_bdi`, `trade_bdi_structural`, `trade_dependency_top`, `trade_bilateral_summary`, `trade_bilateral_final`, `trade_full_period`), toutes validées contre les CSV de référence pandas (écarts nuls à négligeables, hors quelques départages d'égalité déjà documentés). Ces tables -- en particulier `trade_bilateral_final` et `trade_full_period` -- sont la base pour une éventuelle section H combinant analyse globale (F) et bilatérale (G), comme en pandas.

<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">

# H. COMBINED COUNTRY SCORING & FINAL CLASSIFICATION
##### Linking Global Analysis & Bilateral Analysis
</div>

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 1. Objectives & Framework
</div>

#### Objective
Two complementary dimensions are combined — trade power (F) and bilateral
vulnerability (G) — because a country can be commercially strong yet
structurally exposed, or small yet well diversified. Neither dimension
alone provides a complete picture.

Integrate the two analytical dimensions into a unified country scoring framework that captures both
**trade power** and **bilateral vulnerability**.

#### Two Dimensions — Two Questions
| Dimension | Section | Question |
|---|---|---|
| **Trade Power** | F — Global Trade Analysis | How strong and competitive is this country commercially ? |
| **Bilateral Vulnerability** | G — Bilateral Analysis | How exposed and dependent is this country on its partners ? |

#### Combination Logic
A country's final profile results from the **intersection** of these
two dimensions — not their average. A powerful country can be highly
vulnerable; a weak country can be well diversified.
```
Final Score = α × F_structural_score
            + β × G_score_import
            + γ × G_score_export
```

**Weights rationale :**
- F score → export-centric → measures competitive strength
- G import score → supply vulnerability → most critical risk dimension
- G export score → market vulnerability → secondary risk dimension

#### Four Strategic Profiles
| Profile | Trade Power | Bilateral Risk | Label |
|---|---|---|---|
| 🏆 Global Champion | High | Low | Strong and diversified |
| ⚠️ Powerful but Exposed | High | High | Strong but vulnerable |
| 🌱 Diversified Minor | Low | Low | Small but resilient |
| 🔴 Fragile | Low | High | Weak and exposed |

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 2. Data Preparation
</div>

`INNER JOIN` de `analysis.trade_structural` (F) et `analysis.trade_full_period` (G) sur `country_name` -- ne garde que les pays présents dans les deux sections (comme le `how='inner'` pandas).

In [152]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_combined AS
SELECT
    f.country_name,
    f.export_share_structural, f.cagr_export, f.volatility_export, f.negfreq_export, f.coverage_ratio_structural,
    f.structural_score, f.structural_profile, f.size_profile, f.growth_profile, f.stability_profile,
    f.consistency_profile, f.balance_profile, f.meta_profile,
    f.z_cagr_export, f.z_volatility_export, f.z_negfreq_export, f.z_export_share, f.z_coverage_ratio,
    f.z_cagr_export - f.z_volatility_export - f.z_negfreq_export AS resilience_export,
    g.hhi_import, g.hhi_export, g.critical_share, g.dependency_top, g.n_critical, g.bdis,
    g.score_import, g.score_export, g.score_full_periode, g.profile_full_periode,
    g.dominant_profile, g.stability_score, g.convergence_rate, g.drift, g.divergence_freq,
    g.divergence_category, g.top_partner
FROM analysis.trade_structural f
JOIN analysis.trade_full_period g ON f.country_name = g.country_name;

Running query in 'db'

,Success


In [153]:
%%sql
SELECT
    (SELECT COUNT(*) FROM analysis.trade_structural) AS section_f_countries,
    (SELECT COUNT(*) FROM analysis.trade_full_period) AS section_g_countries,
    (SELECT COUNT(*) FROM analysis.trade_combined) AS combined_countries;

Running query in 'db'

,section_f_countries,section_g_countries,combined_countries
0,183,188,183


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 3. Combined Score
</div>

Rank normalization (0→1) is used instead of min-max to ensure equal
contribution from F and G despite their different distributions.
Weights 0.70/0.30 (F/G) reflect that trade power is the primary
dimension, while bilateral resilience is a risk modifier.

`f_norm_rank`/`g_norm_rank` répliquent `scipy.stats.rankdata` (méthode `'average'`, le défaut) : `RANK() OVER (...)` donne le rang **minimum** d'un groupe ex æquo, `RANK() + (COUNT(*) OVER (PARTITION BY ...) - 1) / 2` donne le rang moyen -- identique à `rankdata`. Filtré sur `structural_score`/`score_full_periode` non nuls (`NOT isnan(...)`, cf. cas limite 9.5.2), comme le `dropna` pandas.

In [154]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_combined AS
WITH filtered AS (
    SELECT * FROM analysis.trade_combined
    WHERE structural_score IS NOT NULL
      AND score_full_periode IS NOT NULL AND NOT isnan(score_full_periode)
),
ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY structural_score) AS f_rank_min,
        COUNT(*) OVER (PARTITION BY structural_score) AS f_tie_n,
        RANK() OVER (ORDER BY score_full_periode) AS g_rank_min,
        COUNT(*) OVER (PARTITION BY score_full_periode) AS g_tie_n,
        COUNT(*) OVER () AS n
    FROM filtered
)
SELECT * EXCLUDE (f_rank_min, f_tie_n, g_rank_min, g_tie_n, n),
    ((f_rank_min + (f_tie_n - 1) / 2.0) - 1) / (n - 1) AS f_norm_rank,
    1 - (((g_rank_min + (g_tie_n - 1) / 2.0) - 1) / (n - 1)) AS g_norm_rank
FROM ranked;

Running query in 'db'

,Success


In [155]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_combined AS
SELECT *, 0.70 * f_norm_rank + 0.30 * g_norm_rank AS combined_score
FROM analysis.trade_combined;

Running query in 'db'

,Success


In [156]:
%%sql
SELECT * FROM analysis.trade_combined USING SAMPLE 3;

Running query in 'db'

,country_name,export_share_structural,cagr_export,volatility_export,negfreq_export,coverage_ratio_structural,structural_score,structural_profile,size_profile,growth_profile,...,dominant_profile,stability_score,convergence_rate,drift,divergence_freq,divergence_category,top_partner,f_norm_rank,g_norm_rank,combined_score
0,Jamaica,0.01,0.05,22.89,0.40,0.26,-0.25,Moderate,Small,Weak Grower,...,Vulnerable,1.00,0.90,Stable,0.00,Never,USA,0.26,0.23,0.25
1,Russian Federation,2.23,0.06,25.74,0.43,1.68,0.65,Strong,Leader,Moderate Grower,...,Stable but Concentrated,0.71,1.00,Stable,0.29,Frequent,China,0.93,0.78,0.89
2,Comoros,0.00,0.07,56.57,0.22,0.16,-0.19,Moderate,Small,Moderate Grower,...,Vulnerable,0.78,0.56,Improving,0.22,Frequent,United Arab Emirates,0.40,0.29,0.37


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 4. Final Classification
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.1 Classification
</div>

Thresholds 0.80 / 0.55 / 0.30 are calibrated on the actual score
distribution — not arbitrary.

In [157]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_combined AS
WITH classified AS (
    SELECT *,
        CASE
            WHEN combined_score >= 0.80 THEN 'Global Champion'
            WHEN combined_score >= 0.55 THEN 'Competitive'
            WHEN combined_score >= 0.30 THEN 'Vulnerable'
            ELSE 'Fragile'
        END AS combined_profile
    FROM analysis.trade_combined
)
SELECT *,
    CASE
        WHEN f_norm_rank >= 0.50 AND g_norm_rank >= 0.50 THEN 'Strong & Resilient'
        WHEN f_norm_rank >= 0.50 AND g_norm_rank <  0.50 THEN 'Strong but Exposed'
        WHEN f_norm_rank <  0.50 AND g_norm_rank >= 0.50 THEN 'Diversified Minor'
        ELSE 'Weak & Vulnerable'
    END AS strategic_quadrant_combined
FROM classified;

Running query in 'db'

,Success


In [158]:
%%sql
SELECT combined_profile, COUNT(*) AS n_countries
FROM analysis.trade_combined
GROUP BY combined_profile
ORDER BY n_countries DESC;

Running query in 'db'

,combined_profile,n_countries
0,Competitive,58
1,Vulnerable,57
2,Fragile,45
3,Global Champion,23


In [159]:
%%sql
SELECT strategic_quadrant_combined, COUNT(*) AS n_countries
FROM analysis.trade_combined
GROUP BY strategic_quadrant_combined
ORDER BY n_countries DESC;

Running query in 'db'

,strategic_quadrant_combined,n_countries
0,Strong & Resilient,53
1,Weak & Vulnerable,52
2,Diversified Minor,39
3,Strong but Exposed,39


In [160]:
%%sql
SELECT combined_profile, strategic_quadrant_combined, COUNT(*) AS n
FROM analysis.trade_combined
GROUP BY combined_profile, strategic_quadrant_combined
ORDER BY combined_profile, strategic_quadrant_combined;

Running query in 'db'

,combined_profile,strategic_quadrant_combined,n
0,Competitive,Diversified Minor,7
1,Competitive,Strong & Resilient,30
2,Competitive,Strong but Exposed,21
3,Fragile,Diversified Minor,5
4,Fragile,Weak & Vulnerable,40
5,Global Champion,Strong & Resilient,22
6,Global Champion,Strong but Exposed,1
7,Vulnerable,Diversified Minor,27
8,Vulnerable,Strong & Resilient,1
9,Vulnerable,Strong but Exposed,17


In [161]:
%%sql
SELECT country_name, combined_score, structural_profile, profile_full_periode, combined_profile, strategic_quadrant_combined
FROM analysis.trade_combined
WHERE country_name IN (SELECT UNNEST(getvariable('countries')))
ORDER BY combined_score DESC;

Running query in 'db'

,country_name,combined_score,structural_profile,profile_full_periode,combined_profile,strategic_quadrant_combined
0,China,0.97,Strong,Robust,Global Champion,Strong & Resilient
1,France,0.95,Strong,Robust,Global Champion,Strong & Resilient
2,USA,0.93,Strong,Stable but Concentrated,Global Champion,Strong & Resilient
3,Brazil,0.77,Strong,Stable but Concentrated,Competitive,Strong & Resilient
4,Egypt,0.74,Good,Robust,Competitive,Strong & Resilient
5,Canada,0.68,Strong,Critical Dependency,Competitive,Strong but Exposed
6,Qatar,0.56,Moderate,Stable but Concentrated,Competitive,Diversified Minor
7,Tunisia,0.36,Moderate,Vulnerable,Vulnerable,Weak & Vulnerable


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.2 Distribution
</div>

Déjà montrée juste au-dessus (4.1) -- non répétée.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.3 Ranking
</div>

In [162]:
%%sql
SELECT country_name, combined_score, combined_profile
FROM analysis.trade_combined
WHERE combined_score IS NOT NULL
ORDER BY combined_score DESC
LIMIT 10;

Running query in 'db'

,country_name,combined_score,combined_profile
0,Germany,0.99,Global Champion
1,China,0.97,Global Champion
2,Italy,0.96,Global Champion
3,France,0.95,Global Champion
4,United Kingdom,0.95,Global Champion
5,USA,0.93,Global Champion
6,Spain,0.93,Global Champion
7,Singapore,0.92,Global Champion
8,United Arab Emirates,0.91,Global Champion
9,India,0.91,Global Champion


In [163]:
%%sql
SELECT country_name, combined_score, combined_profile
FROM analysis.trade_combined
WHERE combined_score IS NOT NULL
ORDER BY combined_score ASC
LIMIT 10;

Running query in 'db'

,country_name,combined_score,combined_profile
0,Cabo Verde,0.05,Fragile
1,Tonga,0.05,Fragile
2,Curacao,0.06,Fragile
3,Guinea-Bissau,0.07,Fragile
4,Saint Lucia,0.08,Fragile
5,"China, Macao SAR",0.09,Fragile
6,Saint Vincent and the Grenadines,0.11,Fragile
7,Haiti,0.11,Fragile
8,Sudan,0.14,Fragile
9,Botswana,0.14,Fragile


<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.4 Country Positioning Scatter
</div>

Pas de nouvelle donnée -- `structural_score`/`score_full_periode` déjà dans la table, et le quadrant (4.1) capture déjà le même positionnement F vs G.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.5 Strategic Positioning Matrix
</div>

Idem 4.4 -- `strategic_quadrant_combined` (4.1) est la version tabulaire de ce scatter.

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 4.6 Export Table
</div>

Rien à coder -- `analysis.trade_combined` est déjà persistée.

<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 5. Country Trade Profile Card
</div>

Fiche complète pour le pays choisi (`getvariable('country')`) : 4 blocs de données (Trade Data, Trade Power F, Bilateral Risk G, Combined Positioning) + Forces/Faiblesses générées automatiquement par les mêmes règles que pandas (`UNION ALL` de `CASE WHEN`, une ligne par condition remplie).

In [164]:
%%sql
SELECT year, X, M, trade_balance, export_share, import_share, coverage_ratio, rank
FROM analysis.trade_by_country_year
WHERE country_name = getvariable('country')
ORDER BY year DESC
LIMIT 1;

Running query in 'db'

,year,X,M,trade_balance,export_share,import_share,coverage_ratio,rank
0,2024,2.06T,3.36T,-1.29T,9.61,15.20,0.61,2


In [165]:
%%sql
SELECT structural_score, structural_profile, meta_profile,
       size_profile, export_share_structural,
       balance_profile, coverage_ratio_structural,
       growth_profile, cagr_export,
       stability_profile, volatility_export,
       consistency_profile, negfreq_export
FROM analysis.trade_combined
WHERE country_name = getvariable('country');

Running query in 'db'

,structural_score,structural_profile,meta_profile,size_profile,export_share_structural,balance_profile,coverage_ratio_structural,growth_profile,cagr_export,stability_profile,volatility_export,consistency_profile,negfreq_export
0,3.28,Strong,Established Power,Leader,9.09,Structural Deficit,0.64,Weak Grower,0.04,Moderate,10.97,Inconsistent,0.40


In [166]:
%%sql
SELECT score_full_periode, profile_full_periode, top_partner,
       hhi_import, hhi_export, critical_share, n_critical, drift, stability_score
FROM analysis.trade_combined
WHERE country_name = getvariable('country');

Running query in 'db'

,score_full_periode,profile_full_periode,top_partner,hhi_import,hhi_export,critical_share,n_critical,drift,stability_score
0,0.17,Stable but Concentrated,China,0.08,0.07,0.16,0.91,Stable,0.73


In [167]:
%%sql
SELECT combined_score, combined_profile, strategic_quadrant_combined, f_norm_rank, g_norm_rank
FROM analysis.trade_combined
WHERE country_name = getvariable('country');

Running query in 'db'

,combined_score,combined_profile,strategic_quadrant_combined,f_norm_rank,g_norm_rank
0,0.93,Global Champion,Strong & Resilient,0.99,0.77


In [168]:
%%sql
WITH c AS (
    SELECT * FROM analysis.trade_combined WHERE country_name = getvariable('country')
)
SELECT 'Significant global trade weight' AS strength FROM c WHERE size_profile IN ('Leader', 'Major')
UNION ALL SELECT 'Positive export growth trajectory' FROM c WHERE growth_profile IN ('Strong Grower', 'Moderate Grower')
UNION ALL SELECT 'Favorable trade balance' FROM c WHERE balance_profile IN ('Strong Surplus', 'Slight Surplus')
UNION ALL SELECT 'Stable export performance' FROM c WHERE stability_profile IN ('Stable', 'Moderate')
UNION ALL SELECT 'Consistent growth pattern' FROM c WHERE consistency_profile IN ('Very Consistent', 'Consistent')
UNION ALL SELECT 'Well diversified bilateral trade' FROM c WHERE hhi_import < 0.10 AND hhi_export < 0.10
UNION ALL SELECT 'Low critical partner exposure' FROM c WHERE critical_share < 0.20
UNION ALL SELECT 'Improving bilateral trajectory' FROM c WHERE drift = 'Improving'
UNION ALL SELECT 'Top quartile trade power' FROM c WHERE f_norm_rank >= 0.75
UNION ALL SELECT 'Top quartile bilateral resilience' FROM c WHERE g_norm_rank >= 0.75;

Running query in 'db'

,strength
0,Significant global trade weight
1,Stable export performance
2,Well diversified bilateral trade
3,Low critical partner exposure
4,Top quartile trade power
5,Top quartile bilateral resilience


In [169]:
%%sql
WITH c AS (
    SELECT * FROM analysis.trade_combined WHERE country_name = getvariable('country')
)
SELECT 'Limited global trade weight' AS weakness FROM c WHERE size_profile IN ('Mid', 'Small')
UNION ALL SELECT 'Weak or negative export growth' FROM c WHERE growth_profile IN ('Weak Grower', 'Declining')
UNION ALL SELECT 'Structural trade deficit' FROM c WHERE balance_profile IN ('Slight Deficit', 'Structural Deficit')
UNION ALL SELECT 'High export volatility' FROM c WHERE stability_profile IN ('Volatile', 'Very Volatile')
UNION ALL SELECT 'Inconsistent growth pattern' FROM c WHERE consistency_profile IN ('Inconsistent', 'Very Inconsistent')
UNION ALL SELECT 'High critical partner exposure' FROM c WHERE critical_share >= 0.40
UNION ALL SELECT 'Critical bilateral dependency' FROM c WHERE profile_full_periode = 'Critical Dependency'
UNION ALL SELECT 'Worsening bilateral trajectory' FROM c WHERE drift = 'Worsening'
UNION ALL SELECT 'Bottom quartile trade power' FROM c WHERE f_norm_rank < 0.25
UNION ALL SELECT 'Bottom quartile bilateral resilience' FROM c WHERE g_norm_rank < 0.25;

Running query in 'db'

,weakness
0,Weak or negative export growth
1,Structural trade deficit
2,Inconsistent growth pattern


<div style="background-color: #e9f7ef; padding-left: 20px; border: 1px solid; border-radius: 5px;">

## 6. Stress Testing
</div>

<div style="background-color: #fdedec; padding-left: 20px; border: 1px solid; border-radius: 5px;">

### 6.1 Weight Sensitivity
</div>

`f_norm_rank`/`g_norm_rank` ne changent pas selon le scénario (ce sont les rangs bruts de F et G) -- seul `combined_score` change selon la pondération, donc seul son rang doit être recalculé par scénario. Remplace les 3 graphiques (scatter stabilité, bump chart, heatmap profils) par deux tables de synthèse.

In [170]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_stress_test AS
WITH scenarios(scenario, w_f, w_g) AS (
    VALUES
        ('Baseline (70/30)', 0.70, 0.30),
        ('Balanced (50/50)', 0.50, 0.50),
        ('Trade-heavy (90/10)', 0.90, 0.10),
        ('Resilience-heavy (30/70)', 0.30, 0.70),
        ('Trade-only (100/0)', 1.00, 0.00),
        ('Resilience-only (0/100)', 0.00, 1.00)
),
base AS (
    SELECT country_name, f_norm_rank, g_norm_rank FROM analysis.trade_combined
),
scored AS (
    SELECT s.scenario, s.w_f, s.w_g, b.country_name,
           s.w_f * b.f_norm_rank + s.w_g * b.g_norm_rank AS combined_score
    FROM scenarios s
    CROSS JOIN base b
)
SELECT *,
    RANK() OVER (PARTITION BY scenario ORDER BY combined_score DESC) AS combined_rank,
    CASE
        WHEN combined_score >= 0.80 THEN 'Global Champion'
        WHEN combined_score >= 0.55 THEN 'Competitive'
        WHEN combined_score >= 0.30 THEN 'Vulnerable'
        ELSE 'Fragile'
    END AS profile
FROM scored;

Running query in 'db'

,Success


In [171]:
%%sql
CREATE OR REPLACE TABLE analysis.trade_stress_summary AS
SELECT
    country_name,
    MAX(combined_rank) FILTER (WHERE scenario = 'Baseline (70/30)') AS baseline_rank,
    AVG(combined_rank) AS rank_mean,
    STDDEV(combined_rank) AS rank_std,
    MAX(combined_rank) - MIN(combined_rank) AS rank_range,
    COUNT(DISTINCT profile) - 1 AS n_profile_changes
FROM analysis.trade_stress_test
GROUP BY country_name;

Running query in 'db'

,Success


In [172]:
%%sql
SELECT COUNT(*) AS total_countries FROM analysis.trade_stress_summary;

Running query in 'db'

,total_countries
0,183


In [173]:
%%sql
SELECT country_name, baseline_rank, rank_std, rank_range
FROM analysis.trade_stress_summary
WHERE rank_std < 5
ORDER BY rank_mean
LIMIT 10;

Running query in 'db'

,country_name,baseline_rank,rank_std,rank_range
0,Germany,1,0.82,2
1,Italy,3,2.97,8
2,United Kingdom,5,4.36,12
3,France,4,3.76,10
4,Singapore,8,4.15,10
5,India,10,4.34,10
6,Oman,67,4.59,12
7,Iran,86,3.37,8
8,Mozambique,89,3.62,8
9,Brunei Darussalam,100,3.87,10


In [174]:
%%sql
SELECT country_name, baseline_rank, rank_std, rank_range
FROM analysis.trade_stress_summary
WHERE rank_std > 20
ORDER BY rank_mean
LIMIT 10;

Running query in 'db'

,country_name,baseline_rank,rank_std,rank_range
0,Netherlands,11,21.17,54
1,Rep. of Korea,15,25.90,67
2,Poland,23,22.47,58
3,Viet Nam,24,27.41,70
4,Croatia,47,26.71,63
5,Bulgaria,48,32.12,79
6,Brazil,30,20.48,53
7,Belgium,26,26.63,68
8,Japan,21,40.85,103
9,Australia,27,25.57,66


In [175]:
%%sql
SELECT n_profile_changes, COUNT(*) AS n_countries
FROM analysis.trade_stress_summary
GROUP BY n_profile_changes
ORDER BY n_profile_changes;

Running query in 'db'

,n_profile_changes,n_countries
0,0,57
1,1,73
2,2,44
3,3,9


In [176]:
%%sql
SELECT country_name, scenario, combined_rank, profile
FROM analysis.trade_stress_test
WHERE country_name IN ('USA', 'China', 'France', 'Germany', 'Canada', 'India', 'Japan', 'Brazil', 'Morocco', 'Egypt')
ORDER BY country_name,
    CASE scenario
        WHEN 'Trade-only (100/0)' THEN 1
        WHEN 'Trade-heavy (90/10)' THEN 2
        WHEN 'Baseline (70/30)' THEN 3
        WHEN 'Balanced (50/50)' THEN 4
        WHEN 'Resilience-heavy (30/70)' THEN 5
        WHEN 'Resilience-only (0/100)' THEN 6
    END;

Running query in 'db'

,country_name,scenario,combined_rank,profile
0,Brazil,Trade-only (100/0),27,Global Champion
1,Brazil,Trade-heavy (90/10),26,Global Champion
2,Brazil,Baseline (70/30),30,Competitive
3,Brazil,Balanced (50/50),40,Competitive
4,Brazil,Resilience-heavy (30/70),52,Competitive
5,Brazil,Resilience-only (0/100),79,Competitive
6,Canada,Trade-only (100/0),12,Global Champion
7,Canada,Trade-heavy (90/10),24,Global Champion
8,Canada,Baseline (70/30),52,Competitive
9,Canada,Balanced (50/50),91,Vulnerable


<div style="background-color: #eaf2f8; padding-left: 20px; border: 1px solid; border-radius: 5px;">

# I. Conclusions
</div>

Synthèse courte (pas une reprise du texte pandas -- cf. les chiffres obsolètes déjà trouvés en G.10) :

- **Trois couches, entièrement en SQL** : performance structurelle mondiale (F), vulnérabilité bilatérale (G), score combiné (H) -- reconstruites de A à H sans une seule ligne de pandas pour le calcul, uniquement des requêtes DuckDB.
- **Puissance commerciale et vulnérabilité bilatérale sont indépendantes**, confirmé par nos propres tables (`analysis.trade_combined.strategic_quadrant_combined`) -- un pays peut être fort en F et exposé en G, ou l'inverse.
- **Écarts avec pandas minimes et documentés** : bruit flottant pour l'immense majorité des colonnes, quelques cas limites expliqués (cascade `inf` de Cayman Isds, re-classement Spearman, départages de `MODE()`/profil dominant) -- résolus par choix explicite de l'utilisateur ou identiques là où ça comptait.
- **~20 tables persistées** dans `data/sql/trade_analysis_sql.duckdb`, chacune validée contre les CSV de référence pandas au fil des sections E à H.